# 7. The Labor Market, Wages, and Unemployment

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/quarcs-lab/macro102-site/blob/main/notebooks/ch07-the-labor-market-wages-and-unemployment.ipynb)

## Learning goals

The labor market sets the price of work (the wage) and the amount of work (employment). By the end
of this chapter you will be able to:

- read the main U.S. labor-market statistics: the **employment-population ratio** and the
  **unemployment rate**, and how both move in recessions;
- link the unemployment rate, the participation rate and the employment-population ratio with one identity;
- derive the equilibrium wage and employment of a supply-and-demand model of the labor market, read its
  **comparative statics**, and use it for experiments: an income tax, a fall in labor demand, and a wage
  that does not fall;
- derive the **natural rate of unemployment** from the bathtub model, and the path that unemployment
  follows on its way there;
- derive the **present discounted value** formulas from the geometric series, and use them to value a
  working life and a college degree;
- explain with two labor markets why the college wage premium rose while the supply of college graduates grew;
- load monthly data with pandas, shade recessions, recompute textbook tables, and build interactive
  Plotly figures with sliders, toggles, menus and buttons;
- go beyond the textbook's countries: compare up to 187 countries with a country picker, and spot breaks,
  gaps and implausible values in the data.

## Setup

We first load the Python libraries we need. You only have to run this cell once.

> **Python note.** `import` loads a *library*, a toolbox written by other people. We give each
> library a short nickname, for example `np` for NumPy, so that the code stays short. Text after
> a `#` is a *comment*: Python ignores it, and it is there for human readers. The `try ... except`
> block runs one Colab-only line and simply skips it anywhere else.

In [1]:
# Libraries used in this chapter
import numpy as np                    # numbers and arrays
import pandas as pd                   # data tables
import plotly.express as px           # quick charts from data tables
import plotly.graph_objects as go     # charts built piece by piece (model figures)
import plotly.io as pio
from plotly.subplots import make_subplots       # several panels in one figure
from ipywidgets import interact, interact_manual, FloatSlider, ToggleButtons, Dropdown, SelectMultiple

# One light blue/green look for every chart
BLUE, TEAL, GREEN, GRAY = "#16528c", "#008070", "#7fb23a", "#8a96a3"
pio.templates.default = "plotly_white"
pio.templates["plotly_white"].layout.update(width=860, height=480)   # same size in Colab and on the website
px.defaults.color_discrete_sequence = [BLUE, TEAL, GREEN, "#5aa0d8", "#0b4f47", "#e0a33b"]
WIDE = {"description_width": "initial"}         # show the full slider labels

# Colab only: let Plotly charts appear inside the interactive widgets
try:
    import google.colab
    google.colab.output.enable_custom_widget_manager()
except ImportError:
    pass                                        # not running in Colab: nothing to do

# The data files of this chapter live in the book's GitHub repository
DATA_URL = "https://raw.githubusercontent.com/quarcs-lab/macro102-site/main/data/ch07/"

The next cell defines a helper that only the book website needs. Run it and move on.

In [2]:
# @title Helper for the website versions (run this cell; you do not need to read it)
def slider_figure(make_figure, name, values, start, label=None):
    """Precompute make_figure(name=value) for every value and join the results with one slider.

    The website cannot run Python, so each model figure gets this precomputed version.
    """
    figures = [make_figure(**{name: v}) for v in values]
    first = values.index(start)
    fig = go.Figure(layout=figures[first].layout)
    owner = []                                  # which figure each trace comes from
    for i, f in enumerate(figures):
        for trace in f.data:
            trace.visible = (i == first)
            fig.add_trace(trace)
            owner.append(i)
    steps = []
    for i, f in enumerate(figures):
        layout = {"title": f.layout.title.to_plotly_json(),
                  "annotations": [a.to_plotly_json() for a in f.layout.annotations],
                  "shapes": [s.to_plotly_json() for s in f.layout.shapes]}
        visible = [o == i for o in owner]
        steps.append(dict(method="update", label=str(values[i]), args=[{"visible": visible}, layout]))
    fig.update_layout(sliders=[dict(active=first, steps=steps, pad=dict(t=60),
                                    currentvalue=dict(prefix=(label or name) + " = "))])
    return fig

The multi-country charts have three more website helpers. `region_menu` adds a menu of world regions to a
chart that holds many countries, because a legend with 180 names would be unreadable. `region_groups` lists
the countries that each view of that menu shows together, so that we can give them different colors.
`legend_menu` keeps the textbook's lines on, puts the others in the legend, and adds two reset buttons.

In [3]:
# @title Helper for the website versions of the multi-country charts (run this cell; you do not need to read it)
def region_menu(fig, regions, shown, label=None):
    """Add a region menu: the countries in `shown` stay on, the chosen region's countries go into the legend.

    `regions` maps each country (trace name) to its region. The website cannot run Python, so this menu
    replaces the Colab country picker: pick a region, then click a country in the legend to add it.
    """
    names = [trace.name for trace in fig.data]
    groups = sorted(set(regions[n] for n in names if n in regions))
    buttons = []
    for group in [label or "Textbook: " + ", ".join(shown)] + groups:
        visible = []
        for n in names:
            if n in shown:
                visible.append(True)
            elif regions.get(n) == group:
                visible.append("legendonly")
            else:
                visible.append(False)
        buttons.append(dict(label=group, method="restyle", args=[{"visible": visible}]))
    for trace in fig.data:
        trace.visible = True if trace.name in shown else False
    fig.update_layout(updatemenus=[dict(buttons=buttons, active=0, direction="down", showactive=True,
                                        x=0, xanchor="left", y=1.02, yanchor="bottom")],
                      margin=dict(t=120), legend_title_text="")
    return fig


def region_groups(regions, shown):
    """The views of `region_menu`, for `group_colors`: `shown` alone, then `shown` plus each region's countries."""
    groups = [list(shown)]
    for region in sorted(set(regions.values())):
        groups.append(list(shown) + [n for n in regions if regions[n] == region])
    return groups


def legend_menu(fig, shown, label="Textbook view"):
    """For a chart with up to 40 lines: `shown` stay on, the others wait in the legend; two reset buttons."""
    book = [True if t.name in shown else "legendonly" for t in fig.data]
    everything = [True for t in fig.data]
    for trace, v in zip(fig.data, book):
        trace.visible = v
    fig.update_layout(updatemenus=[dict(type="buttons", direction="right", x=0, xanchor="left", y=1.02,
                                        yanchor="bottom", buttons=[
                                            dict(label=label, method="restyle", args=[{"visible": book}]),
                                            dict(label="All lines", method="restyle", args=[{"visible": everything}])])],
                      margin=dict(t=120), legend_title_text="")
    return fig

## Model overview

The labor market decides two numbers that touch almost every household: how many people have a job and how much a job pays. This chapter first measures them, then explains them with **supply and demand**, the workhorse of economics, now applied to hours of work instead of apples or cars. Two smaller models complete the picture. The **bathtub model** follows the people who lose and find jobs every month and gives the unemployment rate the economy returns to in normal times. **Present discounted value** turns a lifetime of wages into one number today, which lets us ask what a college degree is worth.

This section builds the tools in five blocks. We start with the definitions behind the labor-market statistics. Then we solve the supply-and-demand model and use it for three experiments: an income tax, a fall in labor demand and a wage that does not fall. The third block solves the bathtub model, the fourth derives the present-value formulas, and the fifth combines supply and demand with present values to study the return to education. Each block goes from the assumptions to the solution and then to the predictions. At the end, `### The model in code` turns the key results into Python functions and checks every one of them with numbers.

As in the book, a bar over a letter marks a **parameter**, a number fixed outside the model, such as the labor force $\bar{L}$ or the job finding rate $\bar{f}$. Letters without a bar are **endogenous**: the model solves for them. A star marks an equilibrium or a steady state, as in $w^*$ or $u^*$. Careful: in this chapter $\bar{s}$ is the job *separation* rate, not the investment rate of the Solow model.

### Measuring the labor market

#### Definitions

Split the adult population $N$ (people aged 16 and over) into three groups. The **employed** $E$ have a paid job. The **unemployed** $U$ have no job, have searched for one in the last four weeks, and could start work now. Everybody else is **not in the labor force**: students, retirees, people caring for family, and discouraged workers who stopped searching. The **labor force** is the first two groups together:

$$LF = E + U.$$

Three ratios summarize the market:

| Ratio | Formula | What it measures |
|---|---|---|
| unemployment rate | $u = U/LF$ | the share of people *in the labor force* who look for work and do not have it |
| participation rate | $LF/N$ | the share of adults who work or want to work |
| employment-population ratio | $E/N$ | the share of *all* adults who work |

#### Solving: how the three ratios fit together

The three ratios are linked by an identity. Multiply and divide $E/N$ by the labor force, then use $E = LF - U$:

$$\frac{E}{N} = \frac{E}{LF}\cdot\frac{LF}{N} = \frac{LF - U}{LF}\cdot\frac{LF}{N} = \left(1 - \frac{U}{LF}\right)\frac{LF}{N},$$

so that

$$\frac{E}{N} = (1 - u)\times\frac{LF}{N}. \tag{1}$$

Taking logs turns the product into a sum ([Appendix C.4](https://quarcs-lab.github.io/macro102-site/notebooks/appC-math-review-for-macroeconomics.html#c.4-logarithms)): $\ln(E/N) = \ln(1-u) + \ln(LF/N)$. Since $\ln(1-u) \approx -u$ when $u$ is small ([Appendix C.6](https://quarcs-lab.github.io/macro102-site/notebooks/appC-math-review-for-macroeconomics.html#c.6-approximations)), a one-point rise in the unemployment rate lowers the employment-population ratio by about 1 percent, and a 1 percent rise in participation raises it by 1 percent.

*Intuition.* The employment-population ratio can fall for two different reasons: more of the labor force is jobless, or fewer adults are in the labor force at all. The unemployment rate sees only the first. With 100 adults, a labor force of 62 and 3 unemployed, $u = 3/62 \approx 4.8\%$, participation is $62\%$ and $E/N = (1 - 0.048)\times 0.62 = 0.59$: 59 of the 100 adults work. If one of the 3 jobless people gives up searching, $U$ falls to 2 and $LF$ to 61: the unemployment rate drops to $3.3\%$, yet nobody has found a job and $E/N$ is still 59 percent.

**The kinds of unemployment.** Economists split the unemployment rate into a part that is there in normal times and a part that comes and goes with booms and recessions:

$$u = \underbrace{u_{\text{frictional}} + u_{\text{structural}}}_{\text{natural rate } u^n} + u_{\text{cyclical}}. \tag{2}$$

- **Frictional** unemployment comes from the time it takes to move between jobs, even in a market that works smoothly.
- **Structural** unemployment comes from the rules and institutions of the labor market: hiring and firing costs, unemployment benefits, the minimum wage.
- The **natural rate** $u^n$ is their sum: the rate in an economy that is neither booming nor in recession. The bathtub model below explains its level.
- **Cyclical** unemployment is the rest, $u - u^n$. It is positive in recessions and negative in booms.

#### What the definitions tell us

- **Read the two employment statistics together.** By identity (1), a policy that pushes people out of the labor force lowers $E/N$ but can leave $u$ unchanged, or even lower it. That is why this chapter tracks both statistics (Figs. 1 and 2).
- **A recession shows up in both.** Job losses raise $u$, and by identity (1) they lower $E/N$, roughly one percent for each point of unemployment.
- **The natural rate is not fixed by nature.** By equation (2) it moves with labor-market institutions, which is how the chapter explains Europe's high unemployment (Fig. 7).

### Supply and demand in the labor market

#### Assumptions

There is one kind of labor, measured by employment $L$, and one price for it, the wage $w$. Following the book's diagrams, we put $L$ on the horizontal axis and $w$ on the vertical axis, so each curve gives "the wage at each level of employment".

**1. Labor demand slopes down.** Firms keep adding workers as long as one more worker produces at least what he or she costs. They stop when the **marginal product of labor**, the output added by one more worker, has fallen to the wage. In the production model of [Chapter 4](https://quarcs-lab.github.io/macro102-site/notebooks/ch04-a-model-of-production.html), $Y = \bar{A}\bar{K}^{1/3}L^{2/3}$, this rule is $w = \tfrac{2}{3}\bar{A}\bar{K}^{1/3}L^{-1/3}$: the exponent $-1/3$ makes the wage firms will pay fall as employment rises, because each extra worker adds less output (**diminishing returns**, [Appendix C.7](https://quarcs-lab.github.io/macro102-site/notebooks/appC-math-review-for-macroeconomics.html#c.7-derivatives-and-marginal-thinking)).

**2. Labor supply slopes up.** An hour spent not working costs the wage it could have earned. When the wage rises, leisure becomes more expensive, and people choose to work more.

**3. A straight-line version.** The book draws the curves without equations. To compute with them we use straight lines, the simplest curves with the right slopes:

$$\text{labor demand: } w = \bar{a}_d - \bar{b}_d L, \qquad\qquad \text{labor supply: } w = \bar{a}_s + \bar{b}_s L.$$

The intercept $\bar{a}_d$ is what firms would pay for the very first worker, and $\bar{b}_d$ is how fast that falls as they hire more. The intercept $\bar{a}_s$ is the lowest wage that gets anyone to work, and $\bar{b}_s$ is how much more pay it takes to draw in each extra worker. We assume $\bar{a}_d > \bar{a}_s$, so that the lines cross at positive employment.

**4. The wage clears the market.** The wage adjusts until the amount of work people want to supply equals the amount firms want to hire.

**5. A fixed population.** With $N$ adults, employment also fixes the employment-population ratio $L/N$.

Every diagram of this notebook starts from the same illustrative values: $\bar{a}_d = 100$, $\bar{b}_d = 1$, $\bar{a}_s = 10$ and $\bar{b}_s = 1$, the default positions of the sliders of Figs. 3 to 6.

#### Solving the model

**Step 1: the equilibrium.** In equilibrium both curves give the same wage at the same employment. Set the two right-hand sides equal:

$$\bar{a}_d - \bar{b}_d L^* = \bar{a}_s + \bar{b}_s L^*.$$

Collect the terms with $L^*$ on the right and the constants on the left, then divide by $\bar{b}_d + \bar{b}_s$ ([Appendix C.9](https://quarcs-lab.github.io/macro102-site/notebooks/appC-math-review-for-macroeconomics.html#c.9-solving-equations-and-systems)):

$$\bar{a}_d - \bar{a}_s = (\bar{b}_d + \bar{b}_s)L^* \quad\Longrightarrow\quad L^* = \frac{\bar{a}_d - \bar{a}_s}{\bar{b}_d + \bar{b}_s}.$$

For the wage, put $L^*$ back into the demand curve and write everything over the common denominator:

$$w^* = \bar{a}_d - \bar{b}_d\frac{\bar{a}_d - \bar{a}_s}{\bar{b}_d + \bar{b}_s} = \frac{\bar{a}_d\bar{b}_d + \bar{a}_d\bar{b}_s - \bar{b}_d\bar{a}_d + \bar{b}_d\bar{a}_s}{\bar{b}_d + \bar{b}_s}.$$

The terms $\bar{a}_d\bar{b}_d$ cancel, which leaves

$$L^* = \frac{\bar{a}_d - \bar{a}_s}{\bar{b}_d + \bar{b}_s}, \qquad\qquad w^* = \frac{\bar{b}_s\bar{a}_d + \bar{b}_d\bar{a}_s}{\bar{b}_d + \bar{b}_s}. \tag{3}$$

With our values, $L^* = 90/2 = 45$ and $w^* = (100 + 10)/2 = 55$. The wage is a weighted average of the two intercepts: each intercept is weighted by the slope of the *other* curve.

*Intuition.* Employment depends on the gap between what firms will pay for the first worker and what it takes to get the first person to work, $\bar{a}_d - \bar{a}_s$. Steep curves, large $\bar{b}_d + \bar{b}_s$, close that gap after only a few hires.

**Step 2: an income tax.** Suppose the government taxes labor income at the rate $\tau$. Firms still pay $w$, but workers keep $(1-\tau)w$. People decide how much to work from what they keep, so the supply curve now reads $(1-\tau)w = \bar{a}_s + \bar{b}_s L$. Solved for the wage firms pay, it is $w = (\bar{a}_s + \bar{b}_s L)/(1-\tau)$: at every level of employment, firms must pay more than before to get the same work, so the supply curve shifts up and to the left. To find the new equilibrium, multiply the demand curve by $(1-\tau)$ and set it equal to the taxed supply curve:

$$(1-\tau)(\bar{a}_d - \bar{b}_d L) = \bar{a}_s + \bar{b}_s L \quad\Longrightarrow\quad (1-\tau)\bar{a}_d - \bar{a}_s = \big[(1-\tau)\bar{b}_d + \bar{b}_s\big]L,$$

so that

$$L_\tau = \frac{(1-\tau)\bar{a}_d - \bar{a}_s}{(1-\tau)\bar{b}_d + \bar{b}_s}, \qquad w_\tau = \bar{a}_d - \bar{b}_d L_\tau. \tag{4}$$

With $\tau = 0$ this is equation (3) again. With $\tau = 0.25$ (the default of Fig. 4), $L_\tau = (75 - 10)/(0.75 + 1) = 65/1.75 \approx 37.1$. Firms pay $w_\tau = 100 - 37.1 = 62.9$, and workers keep $0.75 \times 62.9 = 47.1$. The gap between the two, 15.7, is the **tax wedge**: the tax collected on each unit of work.

To see that a higher tax always lowers employment, divide the numerator and denominator of (4) by $1-\tau$:

$$L_\tau = \frac{\bar{a}_d - \bar{a}_s/(1-\tau)}{\bar{b}_d + \bar{b}_s/(1-\tau)}.$$

A higher $\tau$ raises $1/(1-\tau)$. That lowers the numerator and raises the denominator, so $L_\tau$ falls.

**Step 3: a fall in labor demand.** Suppose firms become less willing to hire, for example because new rules make firing costly or an input such as oil becomes dearer. Model it as a downward shift of the demand curve by an amount $\Delta$: the intercept becomes $\bar{a}_d - \Delta$. Equation (3) is linear in the intercepts, so the changes can be read off directly:

$$\Delta L^* = -\frac{\Delta}{\bar{b}_d + \bar{b}_s}, \qquad\qquad \Delta w^* = -\frac{\bar{b}_s\,\Delta}{\bar{b}_d + \bar{b}_s}.$$

With $\Delta = 20$ (the default of Fig. 5), employment falls from 45 to 35 and the wage from 55 to 45. Both fall.

**Step 4: a wage that does not fall.** Now suppose the wage stays at its old level $\bar{w} = w^*$ after demand falls. The market no longer clears: firms hire what they want at $\bar{w}$, which we read off the new demand curve by solving $\bar{w} = (\bar{a}_d - \Delta) - \bar{b}_d L$ for $L$. Since $\bar{a}_d - w^* = \bar{b}_d L^*$ at the old equilibrium,

$$L'' = \frac{\bar{a}_d - \Delta - \bar{w}}{\bar{b}_d} = L^* - \frac{\Delta}{\bar{b}_d}. \tag{5}$$

At the wage $\bar{w}$, however, the supply curve says that $L^*$ people still want to work. The difference, $L^* - L'' = \Delta/\bar{b}_d$, is the number of people who want a job at the going wage but cannot find one. With our values, $L'' = 45 - 20 = 25$, and 20 people are left without a job.

*Intuition.* With a flexible wage, part of the shock is absorbed by a lower wage, and some people choose to work less. With a rigid wage, the whole shock falls on employment. Comparing step 3 with equation (5), the fall in employment is larger by the factor $(\bar{b}_d + \bar{b}_s)/\bar{b}_d$, which is 2 with our slopes.

#### What the model predicts

**Comparative statics.** Each parameter of equation (3) moves one curve:

| A rise in ... | Which curve moves | $L^*$ | $w^*$ | Elasticity of $L^*$ | Elasticity of $w^*$ |
|---|---|---|---|---|---|
| $\bar{a}_d$ (firms value workers more) | demand shifts up | $+$ | $+$ | $\bar{a}_d/(\bar{a}_d - \bar{a}_s) \approx +1.11$ | $\bar{b}_s\bar{a}_d/(\bar{b}_s\bar{a}_d + \bar{b}_d\bar{a}_s) \approx +0.91$ |
| $\bar{a}_s$ (people need more pay to work) | supply shifts up | $-$ | $+$ | $-\bar{a}_s/(\bar{a}_d - \bar{a}_s) \approx -0.11$ | $\bar{b}_d\bar{a}_s/(\bar{b}_s\bar{a}_d + \bar{b}_d\bar{a}_s) \approx +0.09$ |
| $\bar{b}_d$ | demand gets steeper | $-$ | $-$ | $-\bar{b}_d/(\bar{b}_d+\bar{b}_s) = -0.5$ | $\approx -0.41$ |
| $\bar{b}_s$ | supply gets steeper | $-$ | $+$ | $-\bar{b}_s/(\bar{b}_d+\bar{b}_s) = -0.5$ | $\approx +0.41$ |
| $\tau$, equation (4) | supply shifts up and left | $-$ | $+$ (paid), $-$ (kept) | | |

The elasticities come from taking logs of equation (3) ([Appendix C.4](https://quarcs-lab.github.io/macro102-site/notebooks/appC-math-review-for-macroeconomics.html#c.4-logarithms)). For example, $\ln L^* = \ln(\bar{a}_d - \bar{a}_s) - \ln(\bar{b}_d + \bar{b}_s)$, so a 1 percent rise in $\bar{a}_d$, which adds $0.01\,\bar{a}_d$ to the gap $\bar{a}_d - \bar{a}_s$, raises $L^*$ by $\bar{a}_d/(\bar{a}_d - \bar{a}_s)$ percent. The numbers are for our baseline values. The signs hold for any positive values.

- **Demand shocks move the wage and employment in the same direction; supply shocks move them in opposite directions.** This is the key to reading the data: when wages and employment both rise, demand has grown.
- **A labor income tax lowers employment and the employment-population ratio.** Its effect on the unemployment rate is less clear. People who quit because of the tax and stop searching leave the labor force, and by identity (1) they lower $E/N$ but not $u$. People laid off because firms now pay more are unemployed while they search, and if some of them give up, $u$ falls back again.
- **Who bears the tax depends on the slopes.** With equal slopes, the wage firms pay rises by 7.9 and the wage workers keep falls by 7.9: the burden is shared equally. A steeper supply curve puts more of it on workers.
- **Rigid wages amplify shocks.** By equation (5), a wage that does not fall turns the same fall in demand into a larger fall in employment, and it creates people who want to work at the going wage but cannot find a job. This is how Keynes explained the mass unemployment of the 1930s, and rigid prices and wages return in the short-run chapters of the book.

### The bathtub model of unemployment

#### Assumptions

Time runs in months, $t = 0, 1, 2, \dots$

**1. A fixed labor force.** Every member of the labor force $\bar{L}$ is either employed or unemployed:

$$E_t + U_t = \bar{L}.$$

**2. Constant flow rates.** Each month a fixed share $\bar{s}$ of the employed lose their job, the **job separation rate**, and a fixed share $\bar{f}$ of the unemployed find one, the **job finding rate**. Unemployment therefore changes by the inflow minus the outflow:

$$\Delta U_{t+1} \equiv U_{t+1} - U_t = \underbrace{\bar{s}E_t}_{\text{inflow: jobs lost}} - \underbrace{\bar{f}U_t}_{\text{outflow: jobs found}}. \tag{6}$$

The two endogenous variables are $E_t$ and $U_t$, and the two equations above determine them month by month from a starting value $U_0$. In U.S. data about 1 percent of the employed lose their job each month and about a fifth of the unemployed find one, so we use $\bar{s} = 0.01$ and $\bar{f} = 0.20$, the default positions of the sliders of the bathtub chart below.

The name comes from a bathtub with the tap running and the drain open. Job losses are the tap, job findings are the drain, and the number of unemployed is the water in the tub.

#### Solving the model

**Step 1: the steady state.** In a **steady state** unemployment does not change, $\Delta U_{t+1} = 0$, so the inflow equals the outflow. Set (6) to zero and replace $E$ with $\bar{L} - U$ from assumption 1:

$$0 = \bar{s}(\bar{L} - U^*) - \bar{f}U^* = \bar{s}\bar{L} - \bar{s}U^* - \bar{f}U^* = \bar{s}\bar{L} - (\bar{s} + \bar{f})U^*.$$

Move the last term to the left and divide by $\bar{s} + \bar{f}$: $U^* = \bar{s}\bar{L}/(\bar{s} + \bar{f})$. Dividing by the labor force gives the steady-state unemployment rate:

$$u^* \equiv \frac{U^*}{\bar{L}} = \frac{\bar{s}}{\bar{f} + \bar{s}}. \tag{7}$$

With $\bar{s} = 0.01$ and $\bar{f} = 0.20$, $u^* = 0.01/0.21 \approx 0.048$, or 4.8 percent. Because $\bar{s}$ is small next to $\bar{f}$, a quick approximation is $u^* \approx \bar{s}/\bar{f} = 5$ percent. The size of the labor force does not matter: only the two flow rates do.

*Intuition.* The water level in a tub is steady when water drains out as fast as it flows in. A slow tap (few job losses) and a fast drain (jobs found quickly) leave little water in the tub, that is, a low unemployment rate. In the steady state about 0.95 percent of the labor force loses a job each month and the same number finds one: large flows in both directions, with a small stock of unemployed in between.

**Step 2: the path to the steady state.** Divide (6) by $\bar{L}$ to write it in terms of the rate $u_t = U_t/\bar{L}$, and use $E_t/\bar{L} = 1 - u_t$:

$$u_{t+1} - u_t = \bar{s}(1 - u_t) - \bar{f}u_t = \bar{s} - (\bar{s} + \bar{f})u_t.$$

Equation (7) says $\bar{s} = (\bar{s} + \bar{f})u^*$. Substitute it for the first $\bar{s}$:

$$u_{t+1} - u_t = -(\bar{s} + \bar{f})(u_t - u^*).$$

Add $u_t - u^*$ to both sides: $u_{t+1} - u^* = (1 - \bar{s} - \bar{f})(u_t - u^*)$. Each month the gap to the steady state shrinks by the same factor $1 - \bar{s} - \bar{f}$. Applying this $t$ times from the start ([Appendix C.10](https://quarcs-lab.github.io/macro102-site/notebooks/appC-math-review-for-macroeconomics.html#c.10-difference-equations-how-macro-variables-move-over-time)):

$$u_t = u^* + (1 - \bar{s} - \bar{f})^t\,(u_0 - u^*). \tag{8}$$

With our rates the factor is $0.79$: each month the gap closes by 21 percent. The gap halves in $\ln 0.5/\ln 0.79 \approx 2.9$ months. Starting from $u_0 = 10\%$, after a year the gap is $5.24 \times 0.79^{12} \approx 0.31$ points, so $u_{12} \approx 5.1\%$.

*Intuition.* The unemployment rate behaves like the capital stock of the Solow model ([Chapter 5](https://quarcs-lab.github.io/macro102-site/notebooks/ch05-the-solow-growth-model.html)): whatever its starting point, it moves toward a steady state, quickly when it is far away and slowly when it is close. Here the speed is high because flows in and out of unemployment are large compared with the stock.

#### What the model predicts

**Comparative statics.** Take logs of equation (7): $\ln u^* = \ln\bar{s} - \ln(\bar{f} + \bar{s})$.

| A 1% rise in ... | changes $u^*$ by | In the bathtub |
|---|---|---|
| job separation rate $\bar{s}$ | $+\bar{f}/(\bar{f} + \bar{s}) = +(1 - u^*) \approx +0.95\%$ | the tap runs faster |
| job finding rate $\bar{f}$ | $-\bar{f}/(\bar{f} + \bar{s}) = -(1 - u^*) \approx -0.95\%$ | the drain gets wider |
| labor force $\bar{L}$ | $0$ | a bigger tub, same water level |

The elasticity with respect to $\bar{s}$ is $1 - \bar{s}/(\bar{f} + \bar{s})$, because $\bar{s}$ appears in both the numerator and the denominator. With $u^*$ small, both elasticities are close to $\pm 1$: the natural rate is roughly proportional to $\bar{s}/\bar{f}$.

- **Only the two flow rates move the natural rate.** Policies that aim at the natural rate must change how often people lose jobs or how fast the jobless find new ones.
- **Policies can work against themselves.** A firing cost lowers $\bar{s}$, but it also makes firms cautious about hiring, which lowers $\bar{f}$. If only $\bar{s}$ falls, from 0.01 to 0.008, the natural rate falls to $0.008/0.208 \approx 3.85\%$. If $\bar{f}$ also falls, to 0.15, it *rises* to $0.008/0.158 \approx 5.06\%$.
- **Spells are short on average.** If a fifth of the unemployed find a job each month, the average spell lasts $1/\bar{f} = 5$ months.
- **A shock fades fast.** By equation (8), a recession that pushes unemployment above $u^*$ is followed by a quick return, unless the shock also changes $\bar{s}$ or $\bar{f}$.

### Present discounted value

#### Assumptions

- Money can be saved or borrowed at a constant interest rate $R$ per year. One dollar saved today becomes $1 + R$ dollars next year.
- Interest is **compounded**: interest earned in one year earns interest itself in later years.
- Payments arrive at the start of a year. "Year 0" is today.
- There is no risk: every promised payment is made.

#### Solving the model

**Step 1: one future payment.** Suppose we deposit $X$ dollars today. After one year we have $X(1+R)$, after two years $X(1+R)(1+R) = X(1+R)^2$, and after $T$ years $X(1+R)^T$ ([Appendix C.3](https://quarcs-lab.github.io/macro102-site/notebooks/appC-math-review-for-macroeconomics.html#c.3-exponents-and-exponential-growth)). The **present discounted value** $pdv$ of a payment received in $T$ years is the deposit today that grows into exactly that payment: $pdv \times (1+R)^T = \text{future value}$. Divide both sides by $(1+R)^T$:

$$pdv = \frac{\text{future value}}{(1 + R)^T}. \tag{9}$$

For example, at $R = 3\%$, 10,000 dollars received in 10 years are worth $10{,}000/1.03^{10} \approx 7{,}441$ dollars today. Dividing by $(1+R)^T$ is called **discounting**, and $1/(1+R)^T$ is the **discount factor** ([Appendix C.12](https://quarcs-lab.github.io/macro102-site/notebooks/appC-math-review-for-macroeconomics.html#c.12-present-value-and-discounting)).

**Step 2: a stream of payments.** Now a payment of the same size arrives every year for $n$ years, starting today. Value each payment with equation (9) and add them up. To shorten the notation, write $a = 1/(1+R)$, a number between 0 and 1:

$$pdv = \text{payment}\times\left[1 + \frac{1}{1+R} + \frac{1}{(1+R)^2} + \dots + \frac{1}{(1+R)^{n-1}}\right] = \text{payment}\times\left[1 + a + a^2 + \dots + a^{n-1}\right].$$

The last payment arrives in year $n - 1$, because the first one arrives in year 0.

**Step 3: the geometric-series trick.** Call the sum in brackets $S$. Multiply it by $a$; every term moves up by one power:

$$S = 1 + a + a^2 + \dots + a^{n-1}, \qquad aS = a + a^2 + \dots + a^{n-1} + a^n.$$

Subtract the second line from the first. All the middle terms cancel, and only the first term of $S$ and the last term of $aS$ survive: $S - aS = 1 - a^n$. Factor out $S$ and divide by $1 - a$:

$$S = \frac{1 - a^n}{1 - a}, \qquad\text{so}\qquad pdv = \text{payment}\times\frac{1 - \left(\frac{1}{1+R}\right)^n}{1 - \frac{1}{1+R}}. \tag{10}$$

**Step 4: a simpler form.** The denominator is $1 - \frac{1}{1+R} = \frac{1+R-1}{1+R} = \frac{R}{1+R}$, and dividing by it means multiplying by $\frac{1+R}{R}$:

$$pdv = \text{payment}\times\frac{1+R}{R}\left[1 - \frac{1}{(1+R)^n}\right].$$

**Step 5: the value of a working life.** The bracket in equation (10) is the value today of one dollar a year for $n$ years. At $R = 3\%$ for $n = 45$ years, a working life from age 22 to 67: $(1+R)/R = 34.33$ and $1/1.03^{45} \approx 0.264$, so the factor is $34.33 \times (1 - 0.264) \approx 25.25$. Each dollar of yearly pay is worth 25.25 dollars today, so a constant wage of 100,000 dollars, the value used in the chart in Fig. 8's section, is worth about 2.53 million dollars today. The value of this stream of future earnings is the value of a person's **human capital**.

*Intuition.* A dollar far in the future is worth little today, because a smaller deposit today would grow into it. Adding up 45 such dollars gives about 25 dollars, not 45: the later years count for less and less.

#### What the model predicts

- **A higher interest rate lowers present values.** In equation (9), $\ln pdv = \ln(\text{future value}) - T\ln(1+R) \approx \ln(\text{future value}) - TR$ (Appendix C.6). So one more point of interest lowers the value of a payment $T$ years away by about $T$ percent: about 9 percent for a payment in 10 years.
- **Distant payments add little.** Each extra year adds $a^{n}$ times the payment, which shrinks toward zero. Letting $n$ grow without limit in Step 4 gives the value of a payment that lasts forever, $\text{payment}\times(1+R)/R$: 34.33 dollars per dollar at $3\%$. The 45 years of a working life already give 74 percent of that.
- **Any two streams can be compared.** Bring every payment to today with equation (9) and add them up; the stream with the larger total is worth more, however its payments are spread over time.

### The return to education

This block combines the two tools above: supply and demand explain the college wage, and present values tell us what the wage gap is worth.

#### Assumptions

**1. Two labor markets.** High school graduates and college graduates are different kinds of labor, each with its own straight-line supply and demand curves, as in the supply-and-demand block. The high school market keeps the baseline values, so its wage is 55. The college market has demand $w = 90 - 0.4L$ and supply $w = 30 + 0.6L$, the values of Figs. 9 and 13.

**2. The college premium.** The **college wage premium** is how much more a college graduate earns, in percent: $w_{\text{college}}/w_{\text{high school}} - 1$.

**3. Shifts.** Demand for college graduates shifts out (up) by $x_d$, so its intercept becomes $90 + x_d$, and supply shifts out (down) by $x_s$, so its intercept becomes $30 - x_s$.

**4. The decision to study.** Studying takes 4 years. During them a student gives up a high school wage of 60,000 dollars and pays tuition, here an illustrative 15,000 dollars a year. After college, the graduate earns 90,000 dollars instead of 60,000 for 45 years. The interest rate is $R = 3\%$.

#### Solving the model

**Step 1: the college equilibrium.** Equation (3) with $\bar{a}_d = 90$, $\bar{b}_d = 0.4$, $\bar{a}_s = 30$ and $\bar{b}_s = 0.6$ gives $L^* = 60/1 = 60$ and $w^* = (0.6 \times 90 + 0.4 \times 30)/1 = 66$. The premium over the high school wage is $66/55 - 1 = 20\%$.

**Step 2: how the shifts move the college wage.** Equation (3) is linear in the intercepts, so raise $\bar{a}_d$ by $x_d$ and lower $\bar{a}_s$ by $x_s$ and keep only the changes:

$$\Delta L^* = \frac{x_d + x_s}{\bar{b}_d + \bar{b}_s}, \qquad\qquad \Delta w^* = \frac{\bar{b}_s\,x_d - \bar{b}_d\,x_s}{\bar{b}_d + \bar{b}_s}. \tag{11}$$

Both shifts raise employment. For the wage they pull in opposite directions, and the wage rises only if $\bar{b}_s x_d > \bar{b}_d x_s$, that is, if $x_d/x_s > \bar{b}_d/\bar{b}_s$. With slopes 0.4 and 0.6, $\Delta w^* = 0.6x_d - 0.4x_s$. The defaults of Fig. 9, $x_d = 15$ and $x_s = 10$, give $\Delta w^* = 9 - 4 = 5$ and $\Delta L^* = 25$: the college wage rises to 71 while college employment rises to 85, and the premium rises to $71/55 - 1 \approx 29\%$. The 1970s defaults of Fig. 13, $x_d = 10$ and $x_s = 25$, give $\Delta w^* = 6 - 10 = -4$: the wage falls to 62.

**Step 3: what the degree is worth.** Bring every cost and every gain to today with equations (9) and (10). Write $S_n$ for the bracket of equation (10), the value today of one dollar a year for $n$ years. The costs are 75,000 dollars a year (lost wage plus tuition) in years 0 to 3, worth $75{,}000\times S_4$. The gain is the extra 30,000 dollars a year for 45 years. As a stream starting in year 0 it would be worth $30{,}000\times S_{45}$; it starts in year 4 instead, so by equation (9) we discount it four more years. The **net present value** of a degree is

$$NPV = (w_{\text{college}} - w_{\text{high school}})\times S_{45}\times\frac{1}{(1+R)^4} \;-\; (w_{\text{high school}} + \text{tuition})\times S_4. \tag{12}$$

At $R = 3\%$, $S_4 \approx 3.83$, $S_{45} \approx 25.25$ and $1/1.03^4 \approx 0.888$. The gain is worth about $30{,}000 \times 25.25 \times 0.888 \approx 673{,}000$ dollars and the cost about $75{,}000 \times 3.83 \approx 287{,}000$ dollars, so $NPV \approx 386{,}000$ dollars. The interest rate at which $NPV = 0$ is the degree's **rate of return**. The code below finds it: about 8.6 percent a year.

#### What the model predicts

- **More graduates and a higher premium together mean that demand grew faster than supply.** By equation (11), a supply shift alone lowers the college wage. A rising premium during decades of rapid growth in the number of graduates (Fig. 8) requires demand shifts large enough that $x_d/x_s > \bar{b}_d/\bar{b}_s$ (Fig. 9). In the 1970s supply won and the premium fell (Fig. 13).
- **Two candidate demand shifters.** *Skill-biased technical change* (computers and software raise the productivity of skilled workers more) and *globalization* (trade with countries where skilled workers are scarce raises their value, and trade with countries rich in unskilled labor lowers the wages of the unskilled).
- **A degree is a good investment at these numbers.** By equation (12), the gain is more than twice the cost, and the return beats a 3 percent interest rate by a wide margin. A higher interest rate or a smaller premium shrinks the NPV, because the gains come later than the costs.

### The model in code

We now turn the key results into Python functions. Each function's description names the equation it computes. The cells after it check the algebra above with numbers, in the order of the blocks.

> **Python note.** `def` creates a *function*: a named recipe that takes inputs (the *arguments*)
> and gives back a result with `return`. Writing `a_d=100.0` gives the argument a *default value*,
> used whenever we do not pass one. A function can return two results at once, separated by a comma.

In [4]:
def labor_demand(L, a_d=100.0, b_d=1.0):
    """Labor demand: the wage firms are willing to pay when they employ L, w = a_d - b_d L."""
    return a_d - b_d * L


def labor_supply(L, a_s=10.0, b_s=1.0):
    """Labor supply: the wage workers need to receive to supply L, w = a_s + b_s L."""
    return a_s + b_s * L


def equilibrium(a_d=100.0, b_d=1.0, a_s=10.0, b_s=1.0, tau=0.0):
    """Equations (3) and (4): employment L* and the wage firms pay w*, when workers keep (1 - tau) w."""
    L_star = ((1 - tau) * a_d - a_s) / ((1 - tau) * b_d + b_s)
    w_star = labor_demand(L_star, a_d, b_d)
    return L_star, w_star


def demand_at_wage(w, a_d=100.0, b_d=1.0):
    """Equation (5): the employment firms want at a given wage w (labor demand solved for L)."""
    return (a_d - w) / b_d


def supply_at_wage(w, a_s=10.0, b_s=1.0):
    """The number of people who want to work at a given wage w (labor supply solved for L)."""
    return (w - a_s) / b_s

Three more functions hold the closed forms of the bathtub model, of present values and of the value of a degree.
`100_000` is simply 100000: Python ignores the underscore, which only makes large numbers easier to read.

In [5]:
def natural_rate(s=0.01, f=0.20):
    """Equation (7): the steady-state unemployment rate u* = s / (f + s)."""
    return s / (f + s)


def annuity_factor(R, n):
    """Equation (10) for a payment of 1: the value today of 1 dollar a year for n years, starting today."""
    a = 1 / (1 + R)
    return (1 - a ** n) / (1 - a)


def college_npv(R, premium=30_000, cost=75_000):
    """Equation (12): net present value of 4 years of college followed by 45 years of a higher wage."""
    return premium * annuity_factor(R, 45) / (1 + R) ** 4 - cost * annuity_factor(R, 4)

A quick check of the functions against the numbers above.

> **Python note.** `L_star, w_star = equilibrium()` stores the two results in two variables. An
> *f-string* such as `f"L* = {L_star:.1f}"` inserts a value into a piece of text; `:.1f` means "one decimal".

In [6]:
L_star, w_star = equilibrium()
print(f"Equilibrium:            L* = {L_star:.1f}, w* = {w_star:.1f}")
print(f"At L*, demand wage = {labor_demand(L_star):.1f} and supply wage = {labor_supply(L_star):.1f}: the market clears")
L_tax, w_tax = equilibrium(tau=0.25)
print(f"With a 25% income tax:  L' = {L_tax:.2f}, firms pay w' = {w_tax:.2f}, workers keep {0.75 * w_tax:.2f}")

Equilibrium:            L* = 45.0, w* = 55.0
At L*, demand wage = 55.0 and supply wage = 55.0: the market clears
With a 25% income tax:  L' = 37.14, firms pay w' = 62.86, workers keep 47.14


**Check 1: the identity of the labor-market ratios.** With 100 adults, a labor force of 62 and 3 unemployed, both sides of identity (1) must agree. Then one jobless person stops searching.

In [7]:
N, LF, U = 100, 62, 3
E = LF - U
print(f"E/N directly:            {E / N:.4f}")
print(f"(1 - u) x LF/N, eq. (1): {(1 - U / LF) * (LF / N):.4f}   with u = {100 * U / LF:.2f}%")
U, LF = 2, 61                                     # one unemployed person leaves the labor force
print(f"After one person gives up: u = {100 * U / LF:.2f}%, E/N = {(LF - U) / N:.2f} (unchanged)")

E/N directly:            0.5900
(1 - u) x LF/N, eq. (1): 0.5900   with u = 4.84%
After one person gives up: u = 3.28%, E/N = 0.59 (unchanged)


**Check 2: the closed form and the elasticities of equation (3).** The function `equilibrium` solves the model with the tax formula. Equation (3) must give the same numbers, and a 1 percent rise in each parameter must reproduce the elasticities of the table.

> **Python note.** `cases` is a list of pairs: a name and a list of the four parameter values, in the order
> `a_d, b_d, a_s, b_s`. `for name, values in cases` unpacks each pair into two variables, and `values[0]` is the
> first element of the list `values`.

In [8]:
a_d, b_d, a_s, b_s = 100.0, 1.0, 10.0, 1.0
print(f"Equation (3): L* = {(a_d - a_s) / (b_d + b_s):.2f}, w* = {(b_s * a_d + b_d * a_s) / (b_d + b_s):.2f}")
L0, w0 = equilibrium(a_d, b_d, a_s, b_s)
cases = [["a_d", [101.0, 1.0, 10.0, 1.0]], ["a_s", [100.0, 1.0, 10.1, 1.0]],
         ["b_d", [100.0, 1.01, 10.0, 1.0]], ["b_s", [100.0, 1.0, 10.0, 1.01]]]
for name, values in cases:
    L1, w1 = equilibrium(values[0], values[1], values[2], values[3])
    print(f"1% higher {name}: L* {100 * (L1 / L0 - 1):+.3f}%, w* {100 * (w1 / w0 - 1):+.3f}%")

Equation (3): L* = 45.00, w* = 55.00
1% higher a_d: L* +1.111%, w* +0.909%
1% higher a_s: L* -0.111%, w* +0.091%
1% higher b_d: L* -0.498%, w* -0.407%
1% higher b_s: L* -0.498%, w* +0.407%


**Check 3: the tax and the rigid wage.** A higher tax rate must always lower employment, equation (4). With a fall in demand of 20, a rigid wage must cut employment by $\Delta/\bar{b}_d = 20$, twice the flexible fall, equation (5).

In [9]:
for tau in [0.0, 0.1, 0.25, 0.4]:
    L_t, w_t = equilibrium(tau=tau)
    print(f"tau = {tau:.2f}: L = {L_t:.2f}, firms pay {w_t:.2f}, workers keep {(1 - tau) * w_t:.2f}")
L_flex = equilibrium(a_d=80.0)[0]
L_rigid = demand_at_wage(w_star, a_d=80.0)
print(f"Demand falls by 20: flexible wage L' = {L_flex:.1f}, rigid wage L'' = {L_rigid:.1f}")
print(f"Want a job at the old wage but have none: {supply_at_wage(w_star) - L_rigid:.1f}")

tau = 0.00: L = 45.00, firms pay 55.00, workers keep 55.00
tau = 0.10: L = 42.11, firms pay 57.89, workers keep 52.11
tau = 0.25: L = 37.14, firms pay 62.86, workers keep 47.14
tau = 0.40: L = 31.25, firms pay 68.75, workers keep 41.25
Demand falls by 20: flexible wage L' = 35.0, rigid wage L'' = 25.0
Want a job at the old wage but have none: 20.0


**Check 4: the bathtub model.** We start at 10 percent unemployment and apply equation (6) month by month. The simulation must match the closed form (8) and settle at the natural rate (7), where the inflow equals the outflow.

> **Python note.** `range(1, 61)` counts 1, 2, ..., 60, one step per month. Inside the loop, `U = U + ...` replaces the old
> number of unemployed with the new one, and `if t in (3, 12, 60)` is true only in those three months.

In [10]:
s, f, L_bar = 0.01, 0.20, 100.0
u_star = natural_rate(s, f)
U = 10.0                                          # start at 10% unemployment
for t in range(1, 61):
    U = U + s * (L_bar - U) - f * U               # equation (6), with E = L_bar - U
    if t in (3, 12, 60):
        closed = u_star + (1 - s - f) ** t * (0.10 - u_star)          # equation (8)
        print(f"Month {t:2d}: simulated u = {100 * U / L_bar:.4f}%, equation (8) gives {100 * closed:.4f}%")
print(f"Natural rate, equation (7): u* = {100 * u_star:.4f}%")
E_star = L_bar * (1 - u_star)
print(f"Steady-state flows: jobs lost {s * E_star:.3f} = jobs found {f * u_star * L_bar:.3f}")

Month  3: simulated u = 7.3445%, equation (8) gives 7.3445%
Month 12: simulated u = 5.0714%, equation (8) gives 5.0714%
Month 60: simulated u = 4.7619%, equation (8) gives 4.7619%
Natural rate, equation (7): u* = 4.7619%
Steady-state flows: jobs lost 0.952 = jobs found 0.952


**Check 5: the elasticities of the natural rate and the firing-cost example.** A 1 percent rise in $\bar{s}$ or $\bar{f}$ must change $u^*$ by about $\pm(1 - u^*) = \pm 0.95$ percent.

In [11]:
u0 = natural_rate(0.01, 0.20)
print(f"1% higher s: u* changes by {100 * (natural_rate(0.0101, 0.20) / u0 - 1):+.3f}%   (1 - u* = {1 - u0:.3f})")
print(f"1% higher f: u* changes by {100 * (natural_rate(0.01, 0.202) / u0 - 1):+.3f}%")
print(f"Firing costs lower s only:     u* = {100 * natural_rate(0.008, 0.20):.2f}%")
print(f"Firing costs lower s and f:    u* = {100 * natural_rate(0.008, 0.15):.2f}%")

1% higher s: u* changes by +0.952%   (1 - u* = 0.952)
1% higher f: u* changes by -0.943%
Firing costs lower s only:     u* = 3.85%
Firing costs lower s and f:    u* = 5.06%


The change for $\bar{f}$ comes out at $-0.94$ rather than $-0.95$ percent, because an elasticity describes very small
changes and 1 percent is small but not zero.

**Check 6: present values, two ways.** A deposit of $pdv$ from equation (9) must grow back into the future payment. The loop over 45 years must match the formula (10), and the formula must approach $(1+R)/R$ for a very long stream.

In [12]:
R = 0.03
deposit = 10_000 / (1 + R) ** 10                   # equation (9)
print(f"10,000 dollars in 10 years are worth {deposit:,.2f} today; {deposit:,.2f} x 1.03^10 = {deposit * (1 + R) ** 10:,.2f}")
total = 0.0
for t in range(45):                                # one discounted dollar per year, t = 0 is today
    total = total + 1 / (1 + R) ** t
print(f"1 dollar a year for 45 years: loop {total:.4f}, equation (10) {annuity_factor(R, 45):.4f}")
print(f"A working life at 100,000 dollars a year: {100_000 * annuity_factor(R, 45):,.0f} dollars")
print(f"1,000 years: {annuity_factor(R, 1000):.4f}   and (1 + R)/R = {(1 + R) / R:.4f}")
print(f"One more point of interest, payment in 10 years: {100 * (1.03 / 1.04) ** 10 - 100:+.1f}%")

10,000 dollars in 10 years are worth 7,440.94 today; 7,440.94 x 1.03^10 = 10,000.00
1 dollar a year for 45 years: loop 25.2543, equation (10) 25.2543
A working life at 100,000 dollars a year: 2,525,427 dollars
1,000 years: 34.3333   and (1 + R)/R = 34.3333
One more point of interest, payment in 10 years: -9.2%


**Check 7: the college market and the value of a degree.** Equation (11) must agree with the equilibrium function for the shifts of Figs. 9 and 13, and equation (12) gives the net present value of a degree.

In [13]:
w_hs = equilibrium()[1]
w_1 = equilibrium(a_d=90.0, b_d=0.4, a_s=30.0, b_s=0.6)[1]
for x_d, x_s in [[15.0, 10.0], [10.0, 25.0]]:
    w_2 = equilibrium(a_d=90.0 + x_d, b_d=0.4, a_s=30.0 - x_s, b_s=0.6)[1]
    formula = (0.6 * x_d - 0.4 * x_s) / (0.4 + 0.6)                      # equation (11)
    print(f"x_d = {x_d:4.1f}, x_s = {x_s:4.1f}: wage change {w_2 - w_1:+.2f} (equation 11: {formula:+.2f}), "
          f"premium {w_1 / w_hs - 1:.0%} -> {w_2 / w_hs - 1:.0%}")
print(f"NPV of a degree at R = 3%: {college_npv(0.03):,.0f} dollars")

x_d = 15.0, x_s = 10.0: wage change +5.00 (equation 11: +5.00), premium 20% -> 29%
x_d = 10.0, x_s = 25.0: wage change -4.00 (equation 11: -4.00), premium 20% -> 13%
NPV of a degree at R = 3%: 385,997 dollars


**The rate of return on a degree.** We raise the interest rate in small steps until the net present value of equation (12) turns negative.

> **Python note.** A `while` loop repeats its indented lines as long as its condition is true. Here it stops at the first
> interest rate at which the degree no longer pays.

In [14]:
R = 0.03
while college_npv(R) > 0:
    R = R + 0.0001                                 # one hundredth of a percentage point at a time
print(f"The NPV of a degree turns negative at R = {100 * R:.2f}%: this is its rate of return")

The NPV of a degree turns negative at R = 8.59%: this is its rate of return


### Drawing the curves

Three small drawing helpers save us from repeating the same lines in every figure. `add_curve` draws
one curve and writes its name (such as $L^s$) at one end, as the textbook does, and skips the name when it
would sit on top of another curve's name (`avoid`); `add_point` marks an
equilibrium with a dot, a label and dotted lines down to the axes; `market_axes` gives every diagram
the same fixed axes.

> **Python note.** `np.linspace(0, 110, 111)` creates 111 evenly spaced numbers from 0 to 110: the
> employment levels at which we compute each curve. `go.Figure()` starts an empty chart and
> `add_trace` adds one line or set of points to it. `w[inside]` keeps only the points where the
> condition `inside` is true. `row` and `col` are only needed in figures with several panels; `None`
> means "the only panel".

In [15]:
L_grid = np.linspace(0, 110, 111)          # employment levels used to draw every curve (the whole x-axis)


def add_curve(fig, w, label, color, dash="solid", row=None, col=None, start=False, avoid=None):
    """Draw one curve inside the axes and write its label at the end (start=True: the top-left end)."""
    inside = (w >= 0) & (w <= 110)                      # keep the part of the curve inside the axes
    L, w = L_grid[inside], w[inside]
    fig.add_trace(go.Scatter(x=L, y=w, mode="lines", name=label, showlegend=False,
                             line=dict(color=color, width=3, dash=dash)), row=row, col=col)
    if start:                                           # downward curves: label above the top-left end
        spot, anchor = (L[0], w[0]), "bottom"
    else:
        spot, anchor = (L[-1], w[-1]), "middle"
    near = avoid is not None and abs(spot[0] - avoid[0]) < 8 and abs(spot[1] - avoid[1]) < 8
    if label != "" and not near:                        # skip a label that would sit on another one
        fig.add_annotation(x=spot[0], y=spot[1], text=label, showarrow=False, xanchor="left", xshift=5,
                           yanchor=anchor, font=dict(color=color, size=15), row=row, col=col)
    return spot

In [16]:
def add_point(fig, L_point, w_point, label, row=None, col=None):
    """Mark an equilibrium with a dot, a short label and dotted lines down to the axes."""
    fig.add_trace(go.Scatter(x=[0, L_point, L_point], y=[w_point, w_point, 0], mode="lines", showlegend=False,
                             line=dict(color="black", width=1, dash="dot")), row=row, col=col)
    fig.add_trace(go.Scatter(x=[L_point], y=[w_point], mode="markers+text", text=[label], showlegend=False,
                             textposition="top center", textfont=dict(size=15),
                             marker=dict(color="black", size=11)), row=row, col=col)


def market_axes(fig, title, x_title="Employment, L", y_title="Wage, w"):
    """Fixed axes for every labor-market diagram (no tick label in the bottom-left corner) and the title."""
    fig.update_xaxes(title_text=x_title, range=[0, 110], tickvals=[20, 40, 60, 80, 100], zeroline=False)
    fig.update_yaxes(title_text=y_title, range=[0, 110], tickvals=[20, 40, 60, 80, 100], zeroline=False)
    fig.update_layout(title=title)
    return fig

### Shading recessions

Three data figures shade the U.S. recessions, so we write the shading once, here. A small file lists
the start (peak) and end (trough) month of every U.S. recession since the 1950s, as dated by the NBER.

> **Python note.** `pd.read_csv(...)` reads a data file into a *DataFrame*, a table with named
> columns. `parse_dates=[...]` turns text such as `"2020-02-01"` into real dates. A `for` loop repeats
> the indented lines once for each recession: `zip(recessions.peak, recessions.trough)` hands over one
> (peak, trough) pair at a time. The `if` skips recessions that ended before the chart starts, and
> `fig.add_vrect` shades the band between two dates.

In [17]:
recessions = pd.read_csv(DATA_URL + "recessions.csv", parse_dates=["peak", "trough"])


def shade_recessions(fig, start):
    """Figs. 7.1, 7.2 and 7.10: shade every NBER recession (peak to trough) that ends after the date start."""
    for peak, trough in zip(recessions.peak, recessions.trough):
        if trough >= pd.Timestamp(start):
            fig.add_vrect(x0=peak, x1=trough, fillcolor=BLUE, opacity=0.15, line_width=0)


recessions.tail(3)

,peak,trough
8,2001-03-01,2001-11-01
9,2007-12-01,2009-06-01
10,2020-02-01,2020-04-01


### Fixed colors for many countries

Most data figures of this chapter end with a section that goes beyond the textbook's countries, where you
can add any country to the chart. A line should keep its color when you add or remove others, so we give
every country one fixed color. The textbook's countries keep the colors of `BOOK_COLORS`. Two countries that
can appear together (in the same menu view) should not share a color, so each country takes the first color
that its neighbors do not use.

> **Python note.** `PALETTE` starts as a list of Plotly's 24 dark colors, and the loop appends the colors of a
> second palette that are dark enough to read on white. `first_free` returns as soon as it finds a color. If every
> color is already used next to this country, the second loop picks the one used least often (`.count`).

In [18]:
LIGHT = {"#F7E1A0", "#E2E2E2", "#FBE426", "#16FF32", "#1CFFCE", "#2ED9FF"}      # too pale on white
PALETTE = list(px.colors.qualitative.Dark24)
for color in px.colors.qualitative.Alphabet:
    if color not in LIGHT:
        PALETTE.append(color)                       # 24 + 20 = 44 distinct colors
PALETTE.remove("#2E91E5")
PALETTE.append("#2E91E5")                          # this blue is close to the U.S. navy: use it last
BOOK_COLORS = {"United States": BLUE, "France": TEAL, "Japan": GREEN, "Euro area (12 countries)": "#e0a33b"}


def first_free(used, taken):
    """The first PALETTE color not in `used`, if possible one that no country has `taken` yet."""
    for color in PALETTE:
        if color not in used and color not in taken:
            return color
    best = PALETTE[0]                               # every color is used: share the least-used one,
    for color in PALETTE:                           # from the end of the palette (far from the default view)
        if used.count(color) <= used.count(best):
            best = color
    return best

> **Python note.** `fixed or {}` is `fixed` when it is given and an empty dictionary when it is `None`, and
> `dict(...)` copies it, so the original stays unchanged. `continue` skips to the next name. The inner loops
> collect `used`, the colors of every country that shares a group with `name`. Each figure below lists the
> countries of its website's default view first, so they get the first colors.

In [19]:
def group_colors(names, groups, fixed=None):
    """Give each country a color that no other country in its groups uses, as far as the palette allows.

    names: every country; groups: the lists of countries shown together (menu views); fixed: {country: color}
    for the textbook's countries, so the default view keeps the book's colors."""
    colors = dict(fixed or {})
    for name in names:
        if name in colors:
            continue
        used = []                                   # colors of the countries shown together with `name`
        for group in groups:
            if name in group:
                for other in group:
                    if other in colors:
                        used.append(colors[other])
        colors[name] = first_free(used, list(colors.values()))
    return colors

## The U.S. labor market in the data

We start with two monthly series: the share of adults who work and the share of the labor force without a job. Both swing sharply in every recession.

### Fig. 1: The U.S. employment-population ratio

**Goal.** See the long-run trend and the recessions in the share of adults who work.

The **employment-population ratio** is the percentage of the population aged 16 and over that has a
job. The data are monthly, from the Bureau of Labor Statistics. The recessions are shaded with
`shade_recessions` from "Model overview".

> **Python note.** `.tail()` shows the last five rows of a DataFrame.

In [20]:
emp = pd.read_csv(DATA_URL + "fig7-1.csv", parse_dates=["date"])
emp.tail()

,date,emp_pop_ratio
795,2026-05-01,59.2
796,2026-06-01,59.0
797,2026-07-01,58.9
798,2026-08-01,59.1
799,2026-09-01,59.2


September 2025 is followed by November 2025: October 2025 is missing because the survey was not
carried out during that autumn's federal government shutdown. The chart simply joins the two months.

> **Python note.** `px.area` draws a line and fills the area below it. `tickvals` chooses the numbers
> printed on an axis. `rangeslider_visible=True` adds a small copy of the chart below the axis: drag
> its handles to zoom in on a period.

In [21]:
fig = px.area(emp, x="date", y="emp_pop_ratio", hover_name="date",
              labels={"date": "Year", "emp_pop_ratio": "Employment-population ratio (%)"},
              title="More adults work than in 1960, but every recession cuts employment")
fig.update_traces(line=dict(color=TEAL, width=2))
shade_recessions(fig, "1960-01-01")
fig.update_yaxes(range=[50, 66], tickvals=[52, 56, 60, 64])         # no label in the bottom-left corner
fig.update_xaxes(rangeslider_visible=True)
fig.show()

**Sources.** Data: [U.S. Bureau of Labor Statistics, Current Population Survey, Employment-Population Ratio (`EMRATIO`), via FRED](https://fred.stlouisfed.org/series/EMRATIO);
recession dates: [NBER, U.S. Business Cycle Expansions and Contractions](https://www.nber.org/research/data/us-business-cycle-expansions-and-contractions),
via the FRED recession indicator [`USREC`](https://fred.stlouisfed.org/series/USREC).
Prepared files: [`data/ch07/fig7-1.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch07/fig7-1.csv), [`data/ch07/recessions.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch07/recessions.csv)
(see the [data notes](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch07/README.md)).
Chart: inspired by Fig. 7.1 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

**What to notice.**

- The trend rose from about 56 percent in 1960 to a peak of 64.7 percent in April 2000, and it has
  stayed below that peak ever since.
- The ratio falls in every recession (the shaded bands). The falls after 2007 and in spring 2020 are
  by far the deepest.
- The textbook's figure ends in early 2023; our data go to September 2026.

#### Key numbers: the trend and the Great Recession

> **Python note.** `set_index("date")` uses the dates as row labels, so `ratio.loc["1960"]` keeps every
> month of 1960 and `ratio.loc["2009":"2012"]` every month from 2009 to 2012. `.idxmax()` and
> `.idxmin()` give the row label (here, the month) of the largest and smallest value, and `:%B %Y`
> prints a date as "April 2000".

In [22]:
ratio = emp.set_index("date").emp_pop_ratio
crisis = ratio.loc["2009":"2012"]
print(f"1960 average:            {ratio.loc['1960'].mean():.1f}%")
print(f"Peak:                    {ratio.max():.1f}% in {ratio.idxmax():%B %Y}")
print(f"December 2006:           {ratio.loc['2006-12-01']:.1f}%")
print(f"January 2010:            {ratio.loc['2010-01-01']:.1f}%  (lowest 2009-2012: {crisis.min():.1f}% in {crisis.idxmin():%B %Y})")
print(f"Fall, Dec. 2006 to Jan. 2010: {ratio.loc['2006-12-01'] - ratio.loc['2010-01-01']:.1f} points")
print(f"Covid low:               {ratio.min():.1f}% in {ratio.idxmin():%B %Y};  latest: {ratio.iloc[-1]:.1f}%")

1960 average:            56.1%
Peak:                    64.7% in April 2000
December 2006:           63.4%
January 2010:            58.5%  (lowest 2009-2012: 58.2% in November 2010)
Fall, Dec. 2006 to Jan. 2010: 4.9 points
Covid low:               51.2% in April 2020;  latest: 59.2%


The textbook's numbers hold: 56 percent in 1960, 63.4 percent at the end of 2006, and a fall of about
5 points by early 2010 (58.5 percent in January 2010 in today's data, against 58.4 in the textbook;
the low point came later, in November 2010).

#### Key numbers: men and women

The textbook says that the rise in the ratio came entirely from women. A second file splits the ratio
by sex. We average the months of each year to compare years.

> **Python note.** `px.line(..., y=["men", "women"])` draws one line per column, and the legend names
> the columns. `groupby(emp_sex.date.dt.year).mean()` collects the rows of each calendar year and
> averages them, and `.loc[[1960, 2000, 2019, 2025]]` picks four rows by their label.

In [23]:
emp_sex = pd.read_csv(DATA_URL + "emp_pop_by_sex.csv", parse_dates=["date"])
fig = px.line(emp_sex, x="date", y=["men", "women"], hover_name="date",
              labels={"date": "Year", "value": "Employment-population ratio (%)", "variable": "Group"},
              title="Fewer men and many more women work than in the 1950s")
shade_recessions(fig, "1948-01-01")
fig.show()
yearly = emp_sex.groupby(emp_sex.date.dt.year)[["men", "women"]].mean()
print(yearly.loc[[1960, 2000, 2019, 2025]].round(1))
print(f"Women's peak: {emp_sex.women.max():.1f}% in {emp_sex.date[emp_sex.women.idxmax()]:%B %Y}")

       men  women
date             
1960  78.9   35.5
2000  71.9   57.5
2019  66.6   55.4
2025  64.9   54.9
Women's peak: 58.0% in April 2000


**Sources.** Data: U.S. Bureau of Labor Statistics, Current Population Survey, employment-population ratios of
[men (`LNS12300001`)](https://fred.stlouisfed.org/series/LNS12300001) and [women (`LNS12300002`)](https://fred.stlouisfed.org/series/LNS12300002), via FRED;
recession dates: [NBER, U.S. Business Cycle Expansions and Contractions](https://www.nber.org/research/data/us-business-cycle-expansions-and-contractions),
via the FRED recession indicator [`USREC`](https://fred.stlouisfed.org/series/USREC).
Prepared files: [`data/ch07/emp_pop_by_sex.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch07/emp_pop_by_sex.csv), [`data/ch07/recessions.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch07/recessions.csv)
(see the [data notes](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch07/README.md)).
Chart: inspired by Fig. 7.1 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

Women's ratio rose from 35.5 percent in 1960 to a peak of 58.0 percent in April 2000. Men's fell from
78.9 percent in 1960 to 66.6 percent in 2019 and 64.9 percent in 2025. Both numbers match the textbook.

#### Beyond the United States

The International Labour Organization (ILO) estimates the employment-population ratio of almost every
country in the same way, for people aged 15 and over. The World Bank publishes these estimates, which start
in 1991. A wider file holds them, one row per country and year.

In [24]:
world_emp = pd.read_csv(DATA_URL + "emp_pop_countries.csv")
world_emp.head()

,iso3,country,region,year,emp_pop_ratio
0,AFG,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",1991,43.38
1,AFG,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",1992,43.34
2,AFG,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",1993,43.30
3,AFG,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",1994,43.25
4,AFG,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",1995,43.19


These are annual *model estimates*: where a country has no labor survey in a year, the ILO fills the gap
with a statistical model. For the United States they run about 0.7 points below the BLS ratio above,
because the ILO counts people from age 15 (the BLS from 16).

> **Python note.** `.isin(names)` keeps the rows of the chosen countries. `color_discrete_map` gives each
> line its fixed color. A `SelectMultiple` is a list in which you can pick several items: hold Ctrl (Cmd on
> a Mac) while you click. In the check below, `.dropna()` drops the empty cells and `.iloc[[0, 1, -1]]`
> picks rows by position (`-1` is the last row).

In [25]:
EMP_COUNTRIES = sorted(world_emp.country.unique())
EMP_REGIONS = dict(zip(world_emp.country, world_emp.region))
EMP_SHOWN = ["United States", "Germany", "Japan"]           # the website's default view
EMP_COLORS = group_colors(EMP_SHOWN + EMP_COUNTRIES, region_groups(EMP_REGIONS, EMP_SHOWN), BOOK_COLORS)


def emp_pop_countries(names=("United States",)):
    """Fig. 7.1 for any countries: employment-population ratio, ages 15+ (ILO estimates)."""
    chosen = world_emp[world_emp.country.isin(names)]
    if len(chosen) == 0:
        return go.Figure(layout=dict(title="Pick at least one country"))
    last = chosen.year.max()
    latest = chosen[chosen.year == last].set_index("country").emp_pop_ratio
    title = f"Share of adults with a job in {last}: {latest.idxmax()} {latest.max():.1f}%"
    if len(latest) > 1:
        title = title + f", {latest.idxmin()} {latest.min():.1f}%"       # highest first, lowest second
    fig = px.line(chosen, x="year", y="emp_pop_ratio", color="country", hover_name="country",
                  color_discrete_map=EMP_COLORS, title=title,
                  labels={"year": "Year", "emp_pop_ratio": "Employed, % of population 15+", "country": "Country"})
    low, high = 50, 66
    if chosen.emp_pop_ratio.min() < low or chosen.emp_pop_ratio.max() > high:
        low, high = 15, 90                       # one wider range for countries far from the U.S.
    fig.update_yaxes(range=[low, high])
    fig.update_xaxes(range=[1990, 2026], tickvals=[1995, 2000, 2005, 2010, 2015, 2020, 2025])      # no label at the bottom-left corner
    return fig

Below, Colab shows a country picker. On the website, the chart holds every country: **pick a region in the menu,
then click a country in the legend to add it.**

In [26]:
interact(emp_pop_countries,
         names=SelectMultiple(options=EMP_COUNTRIES, value=("United States", "Germany", "Japan"), rows=10,
                              description="countries", style=WIDE));

In [27]:
# @title Website version: Fig. 1 for every country (pick a region, then click a country in the legend)
fig = emp_pop_countries(tuple(EMP_COUNTRIES))
region_menu(fig, EMP_REGIONS, shown=EMP_SHOWN, label="U.S., Germany, Japan")
fig.update_layout(title="Since 2000 the U.S. ratio has fallen, while Germany's and Japan's have risen")
fig.update_yaxes(autorange=True)              # the axis adapts to the countries you add
fig

In [28]:
ratios = world_emp.pivot(index="year", columns="country", values="emp_pop_ratio")
print(ratios.loc[[2000, 2025], ["United States", "Germany", "Japan", "France", "Italy"]])
print(ratios.loc[2025].dropna().sort_values().iloc[[0, 1, 2, -3, -2, -1]])

country  United States  Germany  Japan  France  Italy
year                                                 
2000             63.51    53.40  59.48   49.65  42.64
2025             59.11    58.32  61.89   51.17  46.15
country
São Tomé and Príncipe    21.21
Djibouti                 23.49
Yemen                    27.37
Madagascar               82.90
Solomon Islands          83.09
Qatar                    86.90
Name: 2025, dtype: float64


**Sources.** Data: International Labour Organization, modelled estimates, published in the
[World Bank, World Development Indicators, Employment to population ratio, 15+, total (`SL.EMP.TOTL.SP.ZS`)](https://data.worldbank.org/indicator/SL.EMP.TOTL.SP.ZS).
Prepared file: [`data/ch07/emp_pop_countries.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch07/emp_pop_countries.csv)
(see the [data notes](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch07/README.md)).
Chart: inspired by Fig. 7.1 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

**What to notice.**

- Add **Germany** and **Japan**. Between 2000 and 2025 the U.S. ratio fell from 63.5 to 59.1 percent, while
  Germany's rose from 53.4 to 58.3 and Japan's from 59.5 to 61.9. Japan now employs a larger share of its
  adults than the United States.
- Add **France** and **Italy**: 51.2 and 46.2 percent in 2025, far below the U.S. ratio throughout.
- **Look at your data.** The highest ratios (above 82 percent) are in Qatar, whose residents are mostly
  migrant workers, and in poor countries such as Madagascar and Tanzania, where few people can afford not to
  work. The lowest (21 to 27 percent) are in São Tomé and Príncipe, Djibouti, Yemen and Somalia. Where
  surveys are rare, these numbers come mostly from the ILO's model, so treat them with care.

**Your turn.** By how many points did women's employment-population ratio rise between 1960 and 2000,
and by how many did men's fall? Use the `yearly` table.

<details><summary>Hint</summary>

`yearly.loc[2000] - yearly.loc[1960]` subtracts one row from another, column by column.

</details>

In [29]:
# Your turn: try it here

### Fig. 2: The U.S. unemployment rate

**Goal.** See how unemployment jumps in recessions and falls in recoveries.

A person is **unemployed** when she has no job, has looked for one in the past four weeks, and is
available to work. The **labor force** is everybody who is employed or unemployed, and the
**unemployment rate** is the share of the labor force that is unemployed.

In [30]:
unrate = pd.read_csv(DATA_URL + "fig7-2.csv", parse_dates=["date"])
fig = px.area(unrate, x="date", y="unemployment_rate", hover_name="date",
              labels={"date": "Year", "unemployment_rate": "Unemployment rate (%)"},
              title="Unemployment jumps in every recession and falls slowly afterwards")
fig.update_traces(line=dict(color=BLUE, width=2))
shade_recessions(fig, "1950-01-01")
fig.update_yaxes(range=[0, 16], tickvals=[2, 4, 6, 8, 10, 12, 14])   # no label in the bottom-left corner
fig.update_xaxes(rangeslider_visible=True)
fig.show()

**Sources.** Data: [U.S. Bureau of Labor Statistics, Current Population Survey, Unemployment Rate (`UNRATE`), via FRED](https://fred.stlouisfed.org/series/UNRATE);
recession dates: [NBER, U.S. Business Cycle Expansions and Contractions](https://www.nber.org/research/data/us-business-cycle-expansions-and-contractions),
via the FRED recession indicator [`USREC`](https://fred.stlouisfed.org/series/USREC).
Prepared files: [`data/ch07/fig7-2.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch07/fig7-2.csv), [`data/ch07/recessions.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch07/recessions.csv)
(see the [data notes](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch07/README.md)).
Chart: inspired by Fig. 7.2 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

**What to notice.**

- Every shaded recession brings a jump in unemployment. The highest peaks are late 1982, October 2009
  and April 2020.
- The Covid-19 jump was the largest and the shortest: the rate fell back much faster than after 2009.
- Between recessions, unemployment falls gradually, often for many years.

#### Key numbers: peaks and recoveries

In [31]:
u = unrate.set_index("date").unemployment_rate
print(f"May 2007: {u.loc['2007-05-01']:.1f}%   October 2009: {u.loc['2009-10-01']:.1f}%")
print(f"February 2020: {u.loc['2020-02-01']:.1f}%   April 2020: {u.loc['2020-04-01']:.1f}%")
after_2009 = u.loc["2009-11":]
after_2020 = u.loc["2020-05":]
print(f"First month below 4% after 2009:  {after_2009[after_2009 < 4].index[0]:%B %Y}")
print(f"First month below 4% after 2020:  {after_2020[after_2020 < 4].index[0]:%B %Y}")

May 2007: 4.4%   October 2009: 10.0%
February 2020: 3.5%   April 2020: 14.8%
First month below 4% after 2009:  May 2018
First month below 4% after 2020:  December 2021


The rates match the textbook, except April 2020: 14.8 percent in today's data, against 14.7 in the
textbook (seasonal adjustment is revised every year). After the 2009 peak it took until May 2018,
almost nine years, to get below 4 percent again. After the Covid peak it took until December 2021,
under two years.

#### Table 7.1: the composition of the U.S. labor force

A wider file holds the monthly levels behind the unemployment rate, in millions of people. We rebuild
Table 7.1 for January 2023, the textbook's month, and for the latest month.

> **Python note.** `.loc[["2023-01-01", last]]` picks two rows by their dates, and `.T` *transposes*
> the table: rows become columns, so each month becomes a column.

In [32]:
force = pd.read_csv(DATA_URL + "labor_force.csv", parse_dates=["date"]).set_index("date")
force["unemployment_rate"] = 100 * force.unemployed / force.labor_force
last = force.index[-1]
table = force.loc[[pd.Timestamp("2023-01-01"), last]].T
table.columns = ["January 2023", f"{last:%B %Y}"]
table["Textbook (Jan. 2023)"] = [266, 166, 160, 5.7, 100, 3.4]
table.round(2)

,January 2023,September 2026,Textbook (Jan. 2023)
population,265.96,275.55,266.0
labor_force,165.94,170.26,166.0
employed,160.17,163.15,160.0
unemployed,5.77,7.11,5.7
not_in_labor_force,100.02,105.29,100.0
unemployment_rate,3.48,4.18,3.4


January 2023 agrees with the textbook within rounding, except that today's data count 5.77 million
unemployed (the textbook says 5.7), so the rate is 3.48 rather than 3.4 percent. By the latest month,
the labor force has grown to about 170 million and unemployment to about 4.2 percent.

The textbook also notes that, with about 266 million people aged 16 and over, one point of the
employment-population ratio is 2.7 million jobs:

In [33]:
print(f"1% of the population 16+, January 2023: {0.01 * force.loc['2023-01-01', 'population']:.2f} million people")
print(f"1% of the population 16+, {last:%B %Y}:  {0.01 * force.loc[last, 'population']:.2f} million people")

1% of the population 16+, January 2023: 2.66 million people
1% of the population 16+, September 2026:  2.76 million people


#### Beyond the United States

The OECD publishes *harmonised* unemployment rates: every country counts the unemployed with the same
definition as the United States (no job, looking for one, available to work). A wider file holds the monthly
rates of 34 OECD countries, from the FRED database.

In [34]:
monthly = pd.read_csv(DATA_URL + "unemployment_countries_monthly.csv", parse_dates=["date"])
monthly.groupby("country").date.min().sort_values().tail()

country
Greece        1998-04-01
Switzerland   1999-04-01
Iceland       2003-01-01
Turkey        2005-01-01
Israel        2012-01-01
Name: date, dtype: datetime64[ns]

The series start at different dates: Canada, Japan and the United States in 1955, most of Europe in the
1980s, Germany in 1991 (after reunification), Turkey in 2005 and Israel in 2012. New Zealand and Switzerland
publish only quarterly rates. The U.S. rows are the BLS rate of the chart above.

> **Python note.** `.idxmax()` gives the row label of the largest value, and `.loc[...]` then picks that
> whole row, so `peak.country` and `peak.date` tell us where and when unemployment was highest. `:%B %Y`
> prints a date as "April 2020".

In [35]:
MONTHLY_COUNTRIES = sorted(monthly.country.unique())
MONTHLY_SHOWN = ["United States", "Germany"]                # the website's default view
MONTHLY_COLORS = group_colors(MONTHLY_SHOWN + MONTHLY_COUNTRIES, [MONTHLY_SHOWN, MONTHLY_COUNTRIES], BOOK_COLORS)


def unemployment_countries(names=("United States",)):
    """Fig. 7.2 for OECD countries: monthly harmonised unemployment rates, with the highest one in the title."""
    chosen = monthly[monthly.country.isin(names)]
    if len(chosen) == 0:
        return go.Figure(layout=dict(title="Pick at least one country"))
    peak = chosen.loc[chosen.unemployment_rate.idxmax()]
    fig = px.line(chosen, x="date", y="unemployment_rate", color="country", hover_name="country",
                  color_discrete_map=MONTHLY_COLORS,
                  title=f"Highest rate: {peak.country}, {peak.unemployment_rate:.1f}% in {peak.date:%B %Y}",
                  labels={"date": "Year", "unemployment_rate": "Unemployment rate (%)", "country": "Country"})
    high = 16 if chosen.unemployment_rate.max() <= 16 else 30     # widen only for very high rates
    fig.update_yaxes(range=[0, high])
    fig.update_xaxes(rangeslider_visible=True)
    return fig

Below, Colab shows a country picker. On the website, the chart holds all 34 countries: **click a country in the
legend to add it**, and double-click one to show it alone.

In [36]:
interact(unemployment_countries,
         names=SelectMultiple(options=MONTHLY_COUNTRIES, value=("United States",), rows=10,
                              description="countries", style=WIDE));

In [37]:
# @title Website version: Fig. 2 for 34 OECD countries (click a country in the legend to add it)
fig = unemployment_countries(tuple(MONTHLY_COUNTRIES))
legend_menu(fig, MONTHLY_SHOWN, label="U.S. and Germany")
fig.update_layout(title="Covid-19 sent U.S. unemployment to 14.8%; Germany's barely moved")
fig.update_yaxes(autorange=True)              # the axis adapts to the countries you add
fig

In [38]:
rates = monthly.pivot(index="date", columns="country", values="unemployment_rate")
print(rates.loc["2020-02-01":"2020-05-01", ["United States", "Canada", "Germany", "Japan"]])
print(rates[["Spain", "Greece"]].max(), rates[["Spain", "Greece"]].idxmax(), sep="\n")

country     United States  Canada  Germany  Japan
date                                             
2020-02-01            3.5     5.8      3.3    2.4
2020-03-01            4.4     8.5      3.4    2.5
2020-04-01           14.8    13.7      3.5    2.6
2020-05-01           13.2    14.2      3.7    2.8
country
Spain     26.4
Greece    28.3
dtype: float64
country
Spain    2013-02-01
Greece   2013-07-01
dtype: datetime64[ns]


**Sources.** Data: OECD, Main Economic Indicators, harmonised unemployment rates, via FRED, series `LRHUTTTT{CC}M156S`
(for example [Germany, `LRHUTTTTDEM156S`](https://fred.stlouisfed.org/series/LRHUTTTTDEM156S); [all the series](https://fred.stlouisfed.org/searchresults/?st=LRHUTTTT));
U.S. rows: [U.S. Bureau of Labor Statistics, `UNRATE`, via FRED](https://fred.stlouisfed.org/series/UNRATE).
Prepared file: [`data/ch07/unemployment_countries_monthly.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch07/unemployment_countries_monthly.csv)
(see the [data notes](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch07/README.md)).
Chart: inspired by Fig. 7.2 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

**What to notice.**

- Add **Canada**, **Germany** and **Japan**. From February to April 2020, U.S. unemployment jumped from 3.5
  to 14.8 percent and Canada's from 5.8 to 13.7 percent. Germany's rose only from 3.3 to 3.5 percent and
  Japan's from 2.4 to 2.6: both countries subsidized firms that kept their workers on shorter hours instead
  of laying them off.
- Add **Spain** and **Greece**. After the euro crisis their rates passed 26 percent: 26.4 percent in Spain
  (February 2013) and 28.3 percent in Greece (July 2013), almost twice the U.S. peak of April 2020.

**Your turn.** Suppose that in the latest month the unemployment rate had been 10 percent, with the
same labor force. How many fewer people would have had a job?

<details><summary>Hint</summary>

At 10 percent unemployment, employment is `0.9 * force.loc[last, "labor_force"]`. Compare it with
`force.loc[last, "employed"]`.

</details>

In [39]:
# Your turn: try it here

## Supply and demand in the labor market

Now the model. We draw the labor market, add an income tax, and let labor demand fall, first with a flexible wage and then with a wage that does not adjust.

### Fig. 3: The labor market

**Goal.** Draw the supply-and-demand diagram of the labor market and see how its equilibrium moves.

Labor demand $w = \bar{a}_d - \bar{b}_d L$ slopes down and labor supply $w = \bar{a}_s + \bar{b}_s L$
slopes up. They cross at $(L^*, w^*)$, equation (3). If the population is a fixed number $N$, the model also gives
the employment-population ratio $L^*/N$. When you move a slider, the starting curves stay on the chart
as dashed gray lines, so you can see the shift.

In [40]:
def labor_market(a_d=100.0, b_d=1.0, a_s=10.0, b_s=1.0):
    """Fig. 7.3: labor supply, labor demand and the equilibrium (L*, w*)."""
    L_star, w_star = equilibrium(a_d, b_d, a_s, b_s)
    fig = go.Figure()
    add_curve(fig, labor_supply(L_grid, a_s, b_s), "Lˢ", TEAL)
    add_curve(fig, labor_demand(L_grid, a_d, b_d), "Lᵈ", BLUE, start=True)
    if (a_d, b_d, a_s, b_s) != (100.0, 1.0, 10.0, 1.0):          # keep the starting curves for comparison
        add_curve(fig, labor_supply(L_grid), "", GRAY, "dash")
        add_curve(fig, labor_demand(L_grid), "", GRAY, "dash")
    add_point(fig, L_star, w_star, "")
    return market_axes(fig, f"The labor market clears at w* = {w_star:.1f} and L* = {L_star:.1f}")

> **Python note.** `interact` calls the function and redraws its figure every time you move a
> slider. A `FloatSlider` chooses a number between `min` and `max` in steps of `step`, starting at
> `value`. The `;` at the end only hides an extra line of text.

In [41]:
interact(labor_market,
         a_d=FloatSlider(value=100, min=80, max=120, step=5, description="demand intercept a_d", style=WIDE),
         b_d=FloatSlider(value=1.0, min=0.75, max=1.5, step=0.25, description="demand slope b_d", style=WIDE),
         a_s=FloatSlider(value=10, min=0, max=30, step=5, description="supply intercept a_s", style=WIDE),
         b_s=FloatSlider(value=1.0, min=0.75, max=1.5, step=0.25, description="supply slope b_s", style=WIDE));

In [42]:
# @title Website version: Fig. 3 with a precomputed slider for the demand intercept
slider_figure(labor_market, "a_d", [80, 90, 100, 110, 120], start=100, label="demand intercept a_d")

**Source.** Adapted from Fig. 7.3 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

**What to notice.**

- With our illustrative numbers the market clears at $L^* = 45$ and $w^* = 55$.
- A higher $\bar{a}_d$ (firms value every worker more) shifts demand up: the wage **and** employment rise.
- A higher $\bar{a}_s$ (people need a higher wage before they work) shifts supply up: the wage rises but
  employment **falls**.
- Steeper curves (larger slopes) make employment respond less to a given shift.

**Your turn.** Raise the demand intercept $\bar{a}_d$ from 100 to 110. By how much do $L^*$ and $w^*$
change? Check your answer with `equilibrium(a_d=110)`.

<details><summary>Hint</summary>

With both slopes equal to 1, $L^* = (\bar{a}_d - \bar{a}_s)/2$, so each extra unit of $\bar{a}_d$ adds half a
unit to $L^*$.

</details>

In [43]:
# Your turn: try it here

### Fig. 4: An income tax at rate τ

**Goal.** See how a tax on labor income lowers employment and drives a wedge between the wage
firms pay and the wage workers keep.

With a tax at rate $\tau$, a worker keeps only $(1-\tau)w$ of the wage $w$ the firm pays. To supply the
same amount of work as before, workers now need a higher wage from the firm, $w = (\bar{a}_s + \bar{b}_s L)/(1-\tau)$.
Drawn against the wage firms pay, labor supply shifts **back** (up and to the left). The economy moves
from $A$ to $B$ (equation 4).

In [44]:
def income_tax(tau=0.25, b_s=1.0):
    """Fig. 7.4: a labor income tax tau shifts labor supply back; the economy moves from A to B."""
    L_A, w_A = equilibrium(b_s=b_s)
    L_B, w_B = equilibrium(b_s=b_s, tau=tau)
    kept = (1 - tau) * w_B                                       # what workers keep at B
    fig = go.Figure()
    taxed = add_curve(fig, labor_supply(L_grid, b_s=b_s) / (1 - tau), "Lˢ′", TEAL)
    add_curve(fig, labor_supply(L_grid, b_s=b_s), "Lˢ", GRAY, "dash", avoid=taxed)
    add_curve(fig, labor_demand(L_grid), "Lᵈ", BLUE, start=True)
    add_point(fig, L_A, w_A, "A")
    if tau > 0:                                                  # with no tax, B is the same point as A
        add_point(fig, L_B, w_B, "B")
    fig.add_trace(go.Scatter(x=[L_B, L_B], y=[kept, w_B], mode="lines+markers", name="Tax wedge",
                             showlegend=True, line=dict(color=GREEN, width=6)))
    title = f"τ = {tau:.0%}: L′ = {L_B:.1f}, firms pay w′ = {w_B:.1f}, workers keep {kept:.1f}"
    return market_axes(fig, title, y_title="Wage paid by firms, w")

In [45]:
interact(income_tax,
         tau=FloatSlider(value=0.25, min=0, max=0.5, step=0.05, description="tax rate τ", style=WIDE),
         b_s=FloatSlider(value=1.0, min=0.5, max=2.0, step=0.25, description="supply slope b_s", style=WIDE));

In [46]:
# @title Website version: Fig. 4 with a precomputed slider for the tax rate
slider_figure(income_tax, "tau", [0.0, 0.1, 0.2, 0.25, 0.3, 0.4, 0.5], start=0.25, label="tax rate τ")

**Source.** Adapted from Fig. 7.4 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

**What to notice.**

- With a 25 percent tax, employment falls from 45 to about 37.1. Firms pay more (62.9 instead of 55),
  but workers keep less (47.1). The green bar is the tax per unit of work.
- The textbook draws the taxed supply curve parallel to the old one. With a tax proportional to the
  wage, the new curve is a little steeper: the tax takes more when the wage is higher.
- The supply slope $\bar{b}_s$ matters less here than in Fig. 5. A flatter supply curve means people respond
  more to the wage they keep, but a tax proportional to the wage also shifts a flatter curve up by less. Across
  the slider, the 25 percent tax cuts employment by between 6.4 and 8.1.

**Your turn.** What tax rate cuts employment from 45 to 30? Find it with the slider, then check it with
`equilibrium(tau=...)`.

<details><summary>Hint</summary>

Solve $((1-\tau)\cdot 100 - 10)/((1-\tau) + 1) = 30$ for $(1-\tau)$, or try values of `tau` in steps of 0.05.

</details>

In [47]:
# Your turn: try it here

### Fig. 5: A reduction in labor demand

**Goal.** See what happens to the wage and employment when firms want fewer workers.

New rules that make firing workers costly, or a jump in the price of an input such as oil, can make
firms less willing to hire. In the diagram, labor demand shifts down by an amount we call `shift`:
$w = (\bar{a}_d - \text{shift}) - \bar{b}_d L$. The economy moves from $A$ to $B$.

> **Python note.** `fig.add_annotation(...)` with `showarrow=True` draws an arrow from the point
> (`ax`, `ay`) to the point (`x`, `y`). With `axref="x"` and `ayref="y"`, both points are given in the
> units of the axes.

In [48]:
def demand_fall(shift=20.0, b_s=1.0):
    """Fig. 7.5: labor demand shifts back by shift; the wage and employment both fall (A -> B)."""
    L_A, w_A = equilibrium(b_s=b_s)
    L_B, w_B = equilibrium(a_d=100 - shift, b_s=b_s)
    fig = go.Figure()
    after = add_curve(fig, labor_demand(L_grid, 100 - shift), "Lᵈ′", BLUE, start=True)
    add_curve(fig, labor_demand(L_grid), "Lᵈ", GRAY, "dash", start=True, avoid=after)
    add_curve(fig, labor_supply(L_grid, b_s=b_s), "Lˢ", TEAL)
    add_point(fig, L_A, w_A, "A")
    if shift > 0:                                                # with no fall, B is the same point as A
        add_point(fig, L_B, w_B, "B")
        fig.add_annotation(x=60, y=labor_demand(60, 100 - shift) + 1, ax=60, ay=labor_demand(60) - 1,
                           xref="x", yref="y", axref="x", ayref="y", text="", showarrow=True,
                           arrowhead=3, arrowwidth=2, arrowcolor="black")
    return market_axes(fig, f"Demand falls by {shift:.0f}: w {w_A:.1f} → {w_B:.1f}, L {L_A:.1f} → {L_B:.1f}")

In [49]:
interact(demand_fall,
         shift=FloatSlider(value=20, min=0, max=40, step=5, description="fall in labor demand", style=WIDE),
         b_s=FloatSlider(value=1.0, min=0.5, max=2.0, step=0.25, description="supply slope b_s", style=WIDE));

In [50]:
# @title Website version: Fig. 5 with a precomputed slider for the fall in labor demand
slider_figure(demand_fall, "shift", [0, 10, 20, 30, 40], start=20, label="fall in labor demand")

**Source.** Adapted from Fig. 7.5 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

**What to notice.**

- A fall in demand of 20 lowers the wage from 55 to 45 and employment from 45 to 35: both fall.
- With both slopes equal to 1, the fall is split equally between the wage and employment. A flatter
  supply curve puts more of it on employment; a steeper one puts more of it on the wage.
- The employment-population ratio falls too. Unemployment rises at first, and may come back down as
  some discouraged workers stop looking for a job and leave the labor force.

**Your turn.** Set the supply slope $\bar{b}_s$ to 0.5 and then to 2, with a fall in demand of 20. In
which case does employment fall more? Why?

<details><summary>Hint</summary>

Compare `equilibrium(a_d=80, b_s=0.5)` with `equilibrium(b_s=0.5)`, and do the same for `b_s=2`. A flat
supply curve means many people leave work when the wage drops a little.

</details>

In [51]:
# Your turn: try it here

### Fig. 6: A reduction in labor demand with wage rigidity

**Goal.** See why a wage that does not fall turns a fall in demand into a much larger fall in employment.

Suppose the wage stays at its old level $\bar{w} = w^*$ after labor demand falls. Firms then hire only
what they want at $\bar{w}$, so employment moves along the new demand curve to $C$ (equation 5):

$$L'' = \frac{(\bar{a}_d - \text{shift}) - \bar{w}}{\bar{b}_d}$$

At $\bar{w}$, the number of people who want to work is still $L^*$, so $L^* - L''$ people would like a job
but cannot find one. Choose **rigid** or **flexible** with the buttons to compare $C$ with $B$.

In [52]:
def wage_rigidity(shift=20.0, wage="rigid"):
    """Fig. 7.6: labor demand falls; with a rigid wage w-bar the economy moves to C instead of B."""
    L_A, w_bar = equilibrium()                                    # A: the old equilibrium sets w-bar
    L_B, w_B = equilibrium(a_d=100 - shift)                      # B: the wage is flexible
    L_C = demand_at_wage(w_bar, a_d=100 - shift)                 # C: firms' demand at w-bar
    jobless = supply_at_wage(w_bar) - L_C                        # want to work at w-bar, but no job
    fig = go.Figure()
    after = add_curve(fig, labor_demand(L_grid, 100 - shift), "Lᵈ′", BLUE, start=True)
    add_curve(fig, labor_demand(L_grid), "Lᵈ", GRAY, "dash", start=True, avoid=after)
    add_curve(fig, labor_supply(L_grid), "Lˢ", TEAL)
    add_point(fig, L_A, w_bar, "A")
    if wage == "rigid":
        if shift > 0:                                            # with no fall, C is the same point as A
            add_point(fig, L_C, w_bar, "C")
        fig.add_trace(go.Scatter(x=[L_C, L_A], y=[w_bar, w_bar], mode="lines", name="Want a job, cannot find one",
                                 showlegend=True, line=dict(color=GREEN, width=6)))
        title = f"Rigid wage w̄ = {w_bar:.1f}: L″ = {L_C:.1f} (flexible: L′ = {L_B:.1f}); jobless gap {jobless:.1f}"
    else:
        if shift > 0:                                            # with no fall, B is the same point as A
            add_point(fig, L_B, w_B, "B")
        title = f"Flexible wage: the wage falls to w′ = {w_B:.1f} and employment to L′ = {L_B:.1f}"
    return market_axes(fig, title)

> **Python note.** `ToggleButtons` shows one button per option; the function receives the text of the
> chosen button.

In [53]:
interact(wage_rigidity,
         shift=FloatSlider(value=20, min=0, max=40, step=5, description="fall in labor demand", style=WIDE),
         wage=ToggleButtons(options=["rigid", "flexible"], value="rigid", description="wage"));

In [54]:
# @title Website version: Fig. 6 (rigid wage) with a precomputed slider for the fall in labor demand
slider_figure(wage_rigidity, "shift", [0, 10, 20, 30, 40], start=20, label="fall in labor demand")

**Source.** Adapted from Fig. 7.6 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

**What to notice.**

- With a flexible wage, a fall in demand of 20 moves the economy to $B$: employment 35. With the wage
  stuck at $\bar{w} = 55$ it moves to $C$: employment 25, twice the fall.
- At $C$, 45 people want to work at the going wage but only 25 are hired: a gap of 20. Wage rigidity
  creates unemployment that a flexible wage would not.
- This is the reason Keynes gave for the mass unemployment of the Great Depression, and rigid wages
  return in the short-run chapters of the book.

#### The bathtub model of unemployment

The textbook's Section 7.4 explains the **natural rate of unemployment** with a simple model. The labor
force $\bar{L}$ is fixed, and each month a share $\bar{s}$ of the employed lose their job (the **job
separation rate**) while a share $\bar{f}$ of the unemployed find one (the **job finding rate**),
as in equation (6):

$$E_t + U_t = \bar{L} \qquad\qquad \Delta U_{t+1} = \bar{s}E_t - \bar{f}U_t$$

Unemployment is like water in a bathtub: job losses are the faucet and job findings the drain. The level
stops changing when the two flows are equal, which gives the steady-state rate of equation (7)

$$u^* = \frac{\bar{s}}{\bar{f} + \bar{s}}$$

> **Python note.** `range(61)` counts 0, 1, ..., 60, one step per month. Each pass of the loop
> stores the current rate with `.append` and then applies the two equations above once.
> `interact_manual` adds a button, so nothing happens until you press it.

In [55]:
def bathtub_path(s=0.01, f=0.20):
    """The bathtub model, equations (6) and (7): simulate 60 months, starting at 10% unemployment."""
    L_bar, U = 100.0, 10.0                     # labor force and unemployed (only the ratio matters)
    months, rates = [], []
    for t in range(61):
        months.append(t)
        rates.append(100 * U / L_bar)
        E = L_bar - U                          # the labor force: E + U = L_bar
        U = U + s * E - f * U                  # equation (6)
    u_star = 100 * s / (f + s)                 # equation (7)
    path = pd.DataFrame({"month": months, "rate": rates})
    fig = px.line(path, x="month", y="rate", hover_name="month",
                  labels={"month": "Months from the start", "rate": "Unemployment rate (percent)"},
                  title=f"s = {s:.3f}, f = {f:.2f}: unemployment settles at u* = s/(f+s) = {u_star:.1f}%")
    fig.update_traces(line=dict(color=BLUE, width=3))
    fig.add_hline(y=u_star, line_dash="dash", line_color=GRAY)
    fig.update_xaxes(tickvals=[12, 24, 36, 48, 60])     # one tick a year; no label in the bottom-left corner
    fig.update_yaxes(range=[0, 25])
    return fig

Choose a job separation rate and a job finding rate, then press **Run the bathtub**.

In [56]:
run_bathtub = interact_manual.options(manual_name="Run the bathtub")
run_bathtub(bathtub_path,
            s=FloatSlider(value=0.01, min=0.005, max=0.03, step=0.005, description="job separation rate s", style=WIDE),
            f=FloatSlider(value=0.2, min=0.1, max=0.5, step=0.05, description="job finding rate f", style=WIDE));

In [57]:
# @title Website version: the bathtub model with a precomputed slider for the job finding rate
slider_figure(bathtub_path, "f", [0.1, 0.15, 0.2, 0.3, 0.4, 0.5], start=0.2, label="job finding rate f")

**Source.** Adapted from the bathtub model of unemployment (Section 7.4) in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)
Our own simulation and interactive chart, with our own starting point, layout and code; the textbook draws no figure for it.

With the textbook's monthly rates, $\bar{s} = 0.01$ and $\bar{f} = 0.20$, the natural rate is
$0.01/0.21 \approx 4.8$ percent. Starting from 10 percent, unemployment is already close to it after a
year, because a fifth of the unemployed find a job every month.

In [58]:
for s, f in [[0.01, 0.20], [0.008, 0.20], [0.008, 0.15]]:
    print(f"s = {s:<6} f = {f:<5} natural rate u* = {100 * s / (f + s):.2f}%")

s = 0.01   f = 0.2   natural rate u* = 4.76%
s = 0.008  f = 0.2   natural rate u* = 3.85%
s = 0.008  f = 0.15  natural rate u* = 5.06%


The textbook warns that firing costs cut job losses ($\bar{s}$ falls) but can also make firms slow to
hire ($\bar{f}$ falls). In the second line, firing costs only lower $\bar{s}$, and the natural rate falls to
3.85 percent. In the third line they also lower $\bar{f}$, and the natural rate rises to 5.06 percent.

**Your turn.** Keep $\bar{s} = 0.01$. What job finding rate $\bar{f}$ gives a natural rate of 6.8 percent,
Europe's unemployment rate in 2022 (Fig. 7)? Check it with the formula.

<details><summary>Hint</summary>

Solve $0.068 = 0.01/(\bar{f} + 0.01)$ for $\bar{f}$. Then try that value with `bathtub_path(f=...)`.

</details>

In [59]:
# Your turn: try it here

## Unemployment around the world

Unemployment in Europe was once lower than in the United States and then rose well above it. We compare the three large economies and many more.

### Fig. 7: Unemployment in the United States, Europe and Japan

**Goal.** Compare unemployment in three large economies, and see when Europe's rate rose above the
U.S. rate.

The file has annual rates. "Europe" is the euro area of 12 countries from the European Commission's
AMECO database, which starts in 1960. The textbook uses an OECD series instead, which runs up to
about 1 point higher in the mid-1980s.

In [60]:
unemp = pd.read_csv(DATA_URL + "fig7-7.csv")
unemp = unemp.rename(columns={"unrate_us": "United States", "unrate_europe": "Europe", "unrate_japan": "Japan"})
unemp.head(7)

,year,United States,Europe,Japan
0,1955,4.36667,NaN,2.50833
1,1956,4.12500,NaN,2.29167
2,1957,4.30000,NaN,1.92500
3,1958,6.84167,NaN,2.04167
4,1959,5.45000,NaN,2.22500
5,1960,5.54167,2.398,1.62500
6,1961,6.69167,2.153,1.43333


The empty cells (`NaN`, "not a number") are the years 1955–1959, before the European series starts.

> **Python note.** `unemp[unemp.year == year]` keeps the rows where the condition is true, and
> `.iloc[0]` takes the first of them. `add_vline` draws a vertical line at the chosen year.

In [61]:
def unemployment_in(year=2022):
    """Fig. 7.7: unemployment in the U.S., Europe and Japan, with one year's rates in the title."""
    row = unemp[unemp.year == year].iloc[0]
    fig = px.line(unemp, x="year", y=["United States", "Europe", "Japan"],
                  labels={"year": "Year", "value": "Unemployment rate (percent)", "variable": "Economy"})
    fig.add_vline(x=year, line_dash="dot", line_color=GRAY)
    fig.update_yaxes(range=[0, 14])
    fig.update_layout(title=f"{year}: U.S. {row['United States']:.1f}%, Europe {row['Europe']:.1f}%, "
                            f"Japan {row['Japan']:.1f}%", hovermode="x unified")
    return fig

> **Python note.** A `Dropdown` is a menu: the function receives the option you pick.

In [62]:
interact(unemployment_in,
         year=Dropdown(options=list(range(1960, 2026)), value=2022, description="year", style=WIDE));

In [63]:
# @title Website version: Fig. 7 with a precomputed slider for the year
slider_figure(unemployment_in, "year", [1960, 1975, 1985, 1995, 2013, 2020, 2022, 2025], start=2022, label="year")

**Sources.** Data: [U.S. Bureau of Labor Statistics, Unemployment Rate (`UNRATE`), via FRED](https://fred.stlouisfed.org/series/UNRATE), annual average;
[OECD, harmonised unemployment rate for Japan (`LRHUTTTTJPA156S`), via FRED](https://fred.stlouisfed.org/series/LRHUTTTTJPA156S);
Europe: euro area of 12 countries, [European Commission, AMECO database](https://economy-finance.ec.europa.eu/economic-research-and-databases/economic-databases/ameco-database_en),
unemployment rate (`ZUTN`), which replaces the textbook's OECD series.
Prepared file: [`data/ch07/fig7-7.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch07/fig7-7.csv)
(see the [data notes](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch07/README.md)).
Chart: inspired by Fig. 7.7 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

**What to notice.**

- In 2022, U.S. unemployment was 3.6 percent and Japan's 2.6 percent, as in the textbook. Europe's was
  6.8 percent (the textbook's OECD series: 6.7).
- Until the late 1970s, Europe's rate was below the U.S. rate. It rose steeply between 1975 and 1985 and
  has stayed high since, with a peak of 12.1 percent in 2013.
- Japan's rate stayed low throughout. It was above the U.S. rate only in 1999–2001.

#### Key numbers: Europe before and after 1975

In [64]:
early = unemp[(unemp.year >= 1960) & (unemp.year <= 1975)]
late = unemp[unemp.year >= 1985]
print("Average 1960-1975:", early[["United States", "Europe", "Japan"]].mean().round(2).to_dict())
print("Average 1985-2025:", late[["United States", "Europe", "Japan"]].mean().round(2).to_dict())
print("First year with Europe above the U.S.:", unemp[unemp["Europe"] > unemp["United States"]].year.min())
print("Japan above the U.S. in:", unemp[unemp["Japan"] > unemp["United States"]].year.tolist())

Average 1960-1975: {'United States': 5.21, 'Europe': 2.6, 'Japan': 1.33}
Average 1985-2025: {'United States': 5.74, 'Europe': 9.08, 'Japan': 3.45}
First year with Europe above the U.S.: 1979
Japan above the U.S. in: [1999, 2000, 2001]


Up to 1975, Europe averaged 2.6 percent (the textbook says about 2.5), half the U.S. average. Since
1985 it has averaged about 9 percent, against under 6 percent in the United States. The textbook
explains the change with a mix of **adverse shocks** in the 1970s (the productivity slowdown and oil
prices) and **labor market institutions**, such as generous unemployment benefits, that kept
unemployment high once it had risen. In the bathtub model, both work through a lower job finding rate.

#### Beyond three economies

"Europe" in the textbook's figure is an average. A wider file holds the annual unemployment rates of 42
countries from the same European Commission database (AMECO), for the EU members, other European
countries, and the United States, Japan, Canada, Australia, New Zealand and South Korea, plus the euro area
of 12 countries, the "Europe" line above.

In [65]:
ameco = pd.read_csv(DATA_URL + "unemployment_countries.csv")
ameco[ameco.country == "Germany"].iloc[29:33]

,iso3,country,year,unemployment_rate
726,DEU,Germany,1989,5.6
727,DEU,Germany,1990,4.8
728,DEU,Germany,1991,5.2
729,DEU,Germany,1992,6.2


The rows show a **splice**: up to 1990 "Germany" is West Germany, and from 1991 the reunified country. Many
Central and Eastern European series start only in the 1990s or 2000s, after the end of communism.

> **Python note.** `.loc[1985:2025]` keeps the rows from 1985 to 2025, and `.mean()` averages each column.
> `.sort_values()` sorts the averages, so `.index[0]` and `.index[-1]` are the lowest and the highest.

In [66]:
AMECO_COUNTRIES = sorted(ameco.country.unique())
BOOK_ECONOMIES = ("Euro area (12 countries)", "Japan", "United States")
AMECO_SHOWN = list(BOOK_ECONOMIES) + ["Spain", "South Korea"]       # the website's default view
AMECO_COLORS = group_colors(AMECO_SHOWN + AMECO_COUNTRIES, [AMECO_SHOWN, AMECO_COUNTRIES], BOOK_COLORS)


def unemployment_long_run(names=BOOK_ECONOMIES):
    """Fig. 7.7 for any of 42 countries: annual unemployment, with the averages since 1985 in the title."""
    chosen = ameco[ameco.country.isin(names)]
    if len(chosen) == 0:
        return go.Figure(layout=dict(title="Pick at least one country"))
    wide = chosen.pivot(index="year", columns="country", values="unemployment_rate")
    average = wide.loc[1985:2025].mean().sort_values()
    title = f"Average 1985–2025: {average.index[-1]} {average.iloc[-1]:.1f}%"
    if len(average) > 1:
        title = title + f" (highest), {average.index[0]} {average.iloc[0]:.1f}% (lowest)"
    fig = px.line(chosen, x="year", y="unemployment_rate", color="country", hover_name="country",
                  color_discrete_map=AMECO_COLORS, title=title,
                  labels={"year": "Year", "unemployment_rate": "Unemployment rate (percent)", "country": "Economy"})
    high = 14 if chosen.unemployment_rate.max() <= 14 else 40     # widen only for very high rates
    fig.update_yaxes(range=[0, high])
    fig.update_xaxes(tickvals=[1970, 1980, 1990, 2000, 2010, 2020])      # no label at the bottom-left corner
    fig.update_layout(hovermode="x unified")
    return fig

Below, Colab shows a country picker. On the website, the chart holds all 43 economies, with the textbook's three
shown; **click a country in the legend to add it**.

In [67]:
interact(unemployment_long_run,
         names=SelectMultiple(options=AMECO_COUNTRIES, value=BOOK_ECONOMIES, rows=10,
                              description="economies", style=WIDE));

In [68]:
# @title Website version: Fig. 7 for 43 economies (click a country in the legend to add it)
fig = unemployment_long_run(tuple(AMECO_COUNTRIES))
legend_menu(fig, AMECO_SHOWN, label="Textbook view + Spain, South Korea")
fig.update_layout(title="Since 1985 unemployment averaged 16.6% in Spain, 3.5% in Japan and South Korea",
                  hovermode="closest")
fig.update_yaxes(autorange=True)              # the axis adapts to the countries you add
fig

In [69]:
yearly_rates = ameco.pivot(index="year", columns="country", values="unemployment_rate")
print(yearly_rates.loc[[1975, 1985, 1994, 2013, 2025], ["Spain", "Switzerland", "Norway"]])
print(yearly_rates.loc[1985:2025].mean().sort_values().round(1).iloc[[0, 1, 2, 3, 4, -3, -2, -1]])

country  Spain  Switzerland  Norway
year                               
1975       4.5         0.36     2.3
1985      17.8         0.99     2.6
1994      24.1         4.08     5.4
2013      26.1         4.75     3.9
2025      10.5         4.90     4.5
country
Japan               3.5
South Korea         3.5
Iceland             3.6
Switzerland         3.6
Norway              4.0
Spain              16.6
Montenegro         16.8
North Macedonia    27.4
dtype: float64


**Sources.** Data: [European Commission, AMECO database](https://economy-finance.ec.europa.eu/economic-research-and-databases/economic-databases/ameco-database_en),
unemployment rate, Eurostat definition (series `<country>.1.0.0.0.ZUTN`), 42 countries and the euro area of 12.
Prepared file: [`data/ch07/unemployment_countries.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch07/unemployment_countries.csv)
(see the [data notes](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch07/README.md)).
Chart: inspired by Fig. 7.7 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

**What to notice.**

- Add **Spain**: 4.5 percent in 1975, 17.8 in 1985, 24.1 in 1994 and 26.1 in 2013. Its average since 1985,
  16.6 percent, is the highest of all the large economies in the file.
- Add **Switzerland** and **Norway**: averages of 3.6 and 4.0 percent since 1985, close to Japan's 3.5. "European
  unemployment" is not one number: labor market institutions differ a lot across Europe.
- **Look at your data.** Some values are exactly 0: Luxembourg in 1960–1974, Switzerland in 1970–1973 and
  Lithuania in 1990. An exact 0 usually means that unemployment was barely measured, or rounded down, not that
  nobody was looking for work. Check such values before you average them.

#### Table 7.2: hours worked per person

Hours worked per person combine three things: how many people work, how many weeks they work, and how
many hours a week. A wider file holds this measure from the Penn World Table, as an index with the U.S.
in 2019 = 100. Column `hours_pwt10` uses version 10.01, close to the textbook's version 10.0; column
`hours_pwt11` uses the latest version, 11.0.

> **Python note.** `.pivot(index="country", columns="year", values=...)` turns the long table into a
> wide one, with one row per country and one column per year. `[[1960, 1990, 2019]]` keeps three columns.

In [70]:
hours = pd.read_csv(DATA_URL + "hours_per_person.csv")
countries = ["United States", "Italy", "France", "Germany", "U.K.", "Japan", "South Korea"]
pwt10 = hours.pivot(index="country", columns="year", values="hours_pwt10").loc[countries, [1960, 1990, 2019]]
pwt11 = hours.pivot(index="country", columns="year", values="hours_pwt11").loc[countries, [1960, 1990, 2019, 2023]]
print("PWT 10.01 (the textbook's table):")
print(pwt10.round(0))
print("\nPWT 11.0 (latest version):")
print(pwt11.round(1))
print(f"\nEurope (four countries) in 2019: PWT 10.01 {pwt10.loc['Italy':'U.K.', 2019].mean():.1f}, "
      f"PWT 11.0 {pwt11.loc['Italy':'U.K.', 2019].mean():.1f}")

PWT 10.01 (the textbook's table):
year            1960   1990   2019
country                           
United States   88.0  103.0  100.0
Italy          107.0   87.0   86.0
France         111.0   79.0   75.0
Germany        131.0   93.0   88.0
U.K.           114.0   96.0   96.0
Japan          130.0  125.0  110.0
South Korea     89.0  134.0  122.0

PWT 11.0 (latest version):
year            1960   1990   2019   2023
country                                  
United States   87.1  103.0  100.0  100.1
Italy          101.4   83.1   81.8   86.1
France         108.6   75.6   73.4   76.0
Germany        128.0   90.8   85.8   83.4
U.K.           103.1   86.7   86.7   85.5
Japan          124.5  122.4  102.7  103.6
South Korea     88.6  126.5  118.0  120.4

Europe (four countries) in 2019: PWT 10.01 86.0, PWT 11.0 81.9


Version 10.01 reproduces every cell of the textbook's table. In 2019, Japan worked about 10 percent more
hours per person than the United States, and the four European countries about 86 percent of the U.S.
level on average, while in 1960 all four had worked more than Americans. The latest version revises
several countries down (the U.K. by about 9 points in 2019, Japan by 7), but the ranking is the same.
**When a data source is revised, say which version you used.**

**Your turn.** In which of the seven countries did hours per person fall the most between 1960 and 2019?
Use `pwt10`.

<details><summary>Hint</summary>

`(pwt10[2019] - pwt10[1960]).sort_values()` lists the changes from the largest fall to the largest rise.

</details>

In [71]:
# Your turn: try it here

## The rising return to education

College graduates earn much more than high school graduates, even though there are more of them every year. The data show the gap, and the model explains it with shifts in demand and supply.

### Fig. 8: College versus high school wages and employment

**Goal.** Document a key fact of the labor market: college graduates earn much more than high school
graduates, and the gap has grown while the number of college graduates rose.

The textbook uses private data from Acemoglu and Molina. We use the closest public series, from the
Economic Policy Institute (EPI), based on the Current Population Survey:

- the **college wage premium**: how much more college graduates earn per hour than high school graduates
  with similar gender, race, age and region, from 1979;
- the **share of employment** held by workers with a bachelor's or higher degree, from 1976. This is a
  different measure from the textbook's share of *hours*, which is about twice as high.

In [72]:
college = pd.read_csv(DATA_URL + "fig7-8.csv")
college.tail()

,year,college_premium_log_pct,college_premium_pct,college_share_employment_pct
45,2021,46.90,59.84,40.54
46,2022,47.76,61.23,40.63
47,2023,46.69,59.50,40.93
48,2024,45.66,57.88,41.37
49,2025,43.95,55.19,41.82


The premium comes in two versions. `college_premium_pct` is the percent gap: a value of 60 means college
graduates earn 60 percent more. `college_premium_log_pct` is the same gap in **log points**, $100 \ln(1 + \text{gap})$,
the unit economists usually plot (and the textbook uses). For small gaps the two are close; for large
gaps the log version is smaller. Let us check that the two columns agree:

In [73]:
check = 100 * np.log(1 + college.college_premium_pct / 100)
print(f"Largest difference between the two versions: {(check - college.college_premium_log_pct).abs().max():.2f} log points")

Largest difference between the two versions: 0.01 log points


Before plotting, we give the columns readable names for the legend.

In [74]:
named = college.rename(columns={"college_premium_log_pct": "College wage premium (log points)",
                                "college_share_employment_pct": "Share of employment with a college degree"})
fig = px.line(named, x="year", y=["College wage premium (log points)", "Share of employment with a college degree"],
              hover_name="year", color_discrete_sequence=[BLUE, TEAL],
              labels={"year": "Year", "value": "Percent"},
              title="College graduates became more numerous and better paid at the same time")
fig.update_yaxes(range=[0, 60])
fig.update_layout(legend=dict(orientation="h", y=-0.2, yanchor="top"), legend_title_text="", hovermode="x unified")
fig.show()

**Sources.** Data: Economic Policy Institute, State of Working America Data Library, from the Current Population Survey:
[regression-adjusted hourly wage premium of college graduates](https://data.epi.org/wage_gaps/hourly_wage_premium_college/line/year/national/wage_premium_mean_reg_college/overall) and
[share of employment by education](https://data.epi.org/labor_force/labor_force_emp/line/year/national/dist_share_emp/education). They replace the textbook's private Acemoglu–Molina series.
Prepared file: [`data/ch07/fig7-8.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch07/fig7-8.csv)
(see the [data notes](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch07/README.md)).
Chart: inspired by Fig. 7.8 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

**What to notice.**

- The premium rose steeply in the 1980s and more slowly in the 1990s. Since 2000 it has moved between
  about 44 and 49 log points, and it has slipped since 2020.
- At the same time, the share of workers with a college degree more than doubled. More supply **and** a
  higher price: Fig. 9 shows how supply and demand can produce both.
- The textbook's lines start in 1963 and show a dip in the premium in the 1970s (Fig. 13 explains it).
  The public data start in 1979.

#### Key numbers: the premium and the share

In [75]:
c = college.set_index("year")
print(c.loc[[1979, 1990, 2000, 2020, 2025], ["college_premium_log_pct", "college_share_employment_pct"]].round(1))
print(f"Highest premium: {c.college_premium_log_pct.max():.1f} log points in {c.college_premium_log_pct.idxmax()}")

      college_premium_log_pct  college_share_employment_pct
year                                                       
1979                     25.7                          18.3
1990                     40.7                          23.8
2000                     45.9                          27.6
2020                     48.1                          40.9
2025                     44.0                          41.8
Highest premium: 49.2 log points in 2016


The premium is 48.1 log points in 2020, "nearly 50 percent" as the textbook says; it peaked at 49.2 in
2016 and is 44.0 in 2025. (The chapter summary of the textbook speaks of 50 and 100 percent, which does
not fit its own figure.) The share of employment with a college degree rose from 18.3 percent in 1979 to
40.9 percent in 2020.

#### Beyond the United States

EPI's adjusted premium exists only for the United States. For other countries, the OECD publishes a
simpler measure every year: the earnings of full-time workers aged 25–64 with a bachelor's or higher degree,
relative to those with only upper secondary (high school) education, which counts as 100. It also publishes
the share of adults with such a degree. A wider file holds both for 48 countries, mostly from 2013.

In [76]:
oecd_college = pd.read_csv(DATA_URL + "college_premium_countries.csv")
oecd_college[oecd_college.country == "United States"].tail(4)

,iso3,country,year,relative_earnings,premium_log_points,bachelor_share
471,USA,United States,2022,185.12,61.58,NaN
472,USA,United States,2023,184.77,61.39,40.25
473,USA,United States,2024,183.87,60.91,NaN
474,USA,United States,2025,NaN,NaN,41.11


`premium_log_points` turns the relative earnings into log points, $100 \ln(\text{relative earnings}/100)$,
the unit of the chart above. This measure is not adjusted for age, gender or region, so for the United States
it is about 14 log points above EPI's (61 against 46 in 2024). Compare countries with one measure at a time.
One value was dropped when the file was built: Slovakia's 3.5 for 2013, a unit error in the source.

> **Python note.** `.dropna(subset=[...])` drops the rows where that column is empty.
> `.groupby("country").last()` keeps the last row of each country, its latest year. `pd.concat([...],
> axis=1)` puts columns side by side, matching the rows by their label (here, the country).

In [77]:
PREMIUM_COUNTRIES = sorted(oecd_college.dropna(subset=["premium_log_points"]).country.unique())
PREMIUM_COLORS = group_colors(PREMIUM_COUNTRIES, [PREMIUM_COUNTRIES], BOOK_COLORS)   # "All lines" view


def premium_countries(names=("United States",)):
    """Fig. 7.8 for OECD countries: the college premium (log points, full-time workers 25-64)."""
    chosen = oecd_college[oecd_college.country.isin(names)].dropna(subset=["premium_log_points"])
    if len(chosen) == 0:
        return go.Figure(layout=dict(title="Pick at least one country"))
    latest = chosen.groupby("country").last().premium_log_points.sort_values()
    title = f"College premium, latest year: {latest.index[-1]} {latest.iloc[-1]:.0f} log points"
    if len(latest) > 1:
        title = title + f", {latest.index[0]} {latest.iloc[0]:.0f}"
    fig = px.line(chosen, x="year", y="premium_log_points", color="country", hover_name="country",
                  markers=True, color_discrete_map=PREMIUM_COLORS, title=title,
                  labels={"year": "Year", "premium_log_points": "College premium (log points)", "country": "Country"})
    high = 70 if chosen.premium_log_points.max() <= 70 else 130     # widen only for very high premiums
    fig.update_yaxes(range=[0, high])
    fig.update_xaxes(range=[1999, 2026], tickvals=[2005, 2010, 2015, 2020, 2025])      # no label at the bottom-left corner
    return fig

Below, Colab shows a country picker. On the website, the chart holds all 38 countries with earnings data:
**click a country in the legend to add it.**

In [78]:
interact(premium_countries,
         names=SelectMultiple(options=PREMIUM_COUNTRIES, value=("United States",), rows=10,
                              description="countries", style=WIDE));

In [79]:
# @title Website version: Fig. 8 for 38 countries (click a country in the legend to add it)
fig = premium_countries(tuple(PREMIUM_COUNTRIES))
legend_menu(fig, ["United States"], label="United States only")
fig.update_layout(title="The U.S. college premium is among the highest of the rich countries")
fig.update_yaxes(autorange=True)              # the axis adapts to the countries you add
fig

**Sources.** Data: [OECD, Education at a Glance, via the OECD Data Explorer](https://data-explorer.oecd.org/), dataflow `DF_LSO_EARN_REL_UPPER`
(relative earnings of full-time workers aged 25–64 with a bachelor's or higher degree, upper secondary = 100).
Prepared file: [`data/ch07/college_premium_countries.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch07/college_premium_countries.csv)
(see the [data notes](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch07/README.md)).
Chart: inspired by Fig. 7.8 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

Does a country with more graduates pay them less, as supply and demand suggest? Each point below is one
country: its latest premium against its latest share of adults with a degree.

In [80]:
premium = oecd_college.dropna(subset=["premium_log_points"]).groupby("country").last()
share = oecd_college.dropna(subset=["bachelor_share"]).groupby("country").last()
latest_both = pd.concat([premium.premium_log_points, share.bachelor_share], axis=1).dropna().reset_index()
fig = px.scatter(latest_both, x="bachelor_share", y="premium_log_points", hover_name="country",
                 color_discrete_sequence=[TEAL],
                 labels={"bachelor_share": "Adults 25-64 with a bachelor's or higher degree (%)",
                         "premium_log_points": "College premium (log points)"},
                 title="Where graduates are scarce, they earn much more")
fig.update_traces(marker=dict(size=10))
fig.show()
print(f"{len(latest_both)} countries; correlation: {latest_both.bachelor_share.corr(latest_both.premium_log_points):.2f}")

38 countries; correlation: -0.58


**Sources.** Data: [OECD, Education at a Glance, via the OECD Data Explorer](https://data-explorer.oecd.org/), dataflows `DF_LSO_EARN_REL_UPPER` (relative earnings) and
`DF_LSO_NEAC_DISTR_EA` (share of adults aged 25–64 with a bachelor's or higher degree).
Prepared file: [`data/ch07/college_premium_countries.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch07/college_premium_countries.csv)
(see the [data notes](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch07/README.md)).
Chart: inspired by Fig. 7.8 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

**What to notice.**

- Add **Norway**, **Denmark** and **Sweden**: the lowest premiums, 16 to 26 log points. The U.S. premium,
  61 log points in 2024, is higher than in every other country of the file except Costa Rica, Chile, Brazil
  and South Africa.
- In the scatter, countries with more graduates tend to pay them less (a correlation of −0.58 across 38
  countries). Most of that comes from four countries where graduates are scarce (8 to 26 percent of adults)
  and earn 2.2 to 3.1 times as much as high school graduates: South Africa, Brazil, Chile and Costa Rica.
  Among the other 34 countries the link is weak (−0.20): demand for skills differs as much as supply, as
  Fig. 9 explains.

#### Key numbers: present discounted value

To value a college degree we need a tool from the textbook's Section 7.6. A payment received $T$ years
from now is worth less than the same payment today, because money today can earn interest at rate $R$.
Its **present discounted value** is (equation 9)

$$pdv = \frac{\text{future value}}{(1 + R)^T}$$

For a payment that arrives every year for $n$ years, starting today, we add up the present values of
all the payments. The geometric-series formula gives the sum directly (equation 10):

$$pdv = \text{payment} \times \frac{1 - \left(\frac{1}{1+R}\right)^n}{1 - \frac{1}{1+R}}$$

> **Python note.** The loop in `pdv` adds one discounted payment per year, so we can check the formula.
> `**` is a power: `(1 + R) ** t` is $(1+R)^t$. `100_000` is simply 100000; Python ignores the
> underscore, which only makes large numbers easier to read. `:,.0f` prints a number with thousands
> separators and no decimals.

In [81]:
def pdv(payment, R, years):
    """Equation (10): present value of `payment` received every year for `years` years, starting today."""
    total = 0.0
    for t in range(years):                       # t = 0 is today, t = 1 next year, ...
        total = total + payment / (1 + R) ** t
    return total


print(f"$1,000 in 5 years at R = 10%:          ${1000 / 1.10 ** 5:,.0f}")
print(f"$100 a year for 20 years at R = 10%:   ${pdv(100, 0.10, 20):,.0f}")
a = 1 / 1.10
print(f"... and with the formula (10):         ${100 * (1 - a ** 20) / (1 - a):,.0f}")
print(f"$100,000 a year for 45 years at 3%:   ${pdv(100_000, 0.03, 45):,.0f}")
print(f"$30,000 a year for 45 years at 3%:    ${pdv(30_000, 0.03, 45):,.0f}")

$1,000 in 5 years at R = 10%:          $621
$100 a year for 20 years at R = 10%:   $936
... and with the formula (10):         $936
$100,000 a year for 45 years at 3%:   $2,525,427
$30,000 a year for 45 years at 3%:    $757,628


The loop and the formula agree. A prize of \$1,000 in five years is worth \$621 today, and \$100 a year for
20 years is worth \$936, so the second prize is better. A working life of 45 years at \$100,000 a year is
worth about \$2.53 million today: in this sense a typical new worker is a millionaire. And the \$30,000 a
year that a college degree adds (\$90,000 instead of \$60,000) is worth about \$758,000, far more than four
years of tuition and lost earnings.

The chart shows why: each bar is one year's \$100,000 wage valued today. Distant wages count for less, but
they still add up. (`np.arange(45)` gives the whole numbers 0, 1, ..., 44.)

In [82]:
years = np.arange(45)
today = pd.DataFrame({"year": years, "value": 100_000 / 1.03 ** years})
fig = px.bar(today, x="year", y="value", hover_name="year", color_discrete_sequence=[TEAL],
             labels={"year": "Years from today", "value": "Value today (dollars)"},
             title=f"45 years of 100,000-dollar wages at R = 3% are worth {today.value.sum():,.0f} dollars today")
fig.update_xaxes(tickvals=[10, 20, 30, 40])            # no label in the bottom-left corner
fig.show()

**Source.** Our own calculation with equation (10) and illustrative numbers (a wage of 100,000 dollars, $R$ = 3 percent); the
chart uses no data. Inspired by the present-value discussion in Section 7.6 of Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)
To compare with real earnings by education, see the
[U.S. Bureau of Labor Statistics, Education pays](https://www.bls.gov/emp/chart-unemployment-earnings-education.htm).

**Your turn.** Recompute the value of the 45-year working life at $R = 0.01$ and at $R = 0.05$. Why does a
higher interest rate lower the value?

<details><summary>Hint</summary>

Call `pdv(100_000, 0.01, 45)` and `pdv(100_000, 0.05, 45)`. A higher $R$ means a dollar saved today grows
faster, so you need fewer dollars today to match a future payment.

</details>

In [83]:
# Your turn: try it here

### Fig. 9: Understanding the rising return to education

**Goal.** See how the college wage can rise even though many more people go to college.

Fig. 8 showed both more college graduates and a higher college premium. In a supply-and-demand
diagram, more supply alone would *lower* the college wage. So demand for college graduates must have
shifted out too, and by more than supply. Panel (a) is the market for high school graduates, which we
keep unchanged. Panel (b) is the market for college graduates, with demand
$w = (90 + \text{shift}_d) - 0.4L$ and supply $w = (30 - \text{shift}_s) + 0.6L$. Before the shifts, the
college wage is 66 and the high school wage 55, a premium of 20 percent.

We draw the college market in a function of its own, because Fig. 13 uses it again. The curves before
the shifts are dashed gray ($L^s$, $L^d$) and the curves after them solid ($L^{s\prime}$, $L^{d\prime}$). The
equilibrium before the shifts is marked $*$ and the one after them $**$.

> **Python note.** `a or b` is true when at least one of the two conditions is true.

In [84]:
def college_market(fig, shift_d, shift_s, row=None, col=None):
    """Figs. 7.9b and 7.13: the college labor market before (*) and after (**) both curves shift."""
    L_1, w_1 = equilibrium(a_d=90.0, b_d=0.4, a_s=30.0, b_s=0.6)
    L_2, w_2 = equilibrium(a_d=90.0 + shift_d, b_d=0.4, a_s=30.0 - shift_s, b_s=0.6)
    supply = add_curve(fig, labor_supply(L_grid, 30.0 - shift_s, 0.6), "Lˢ′", TEAL, "solid", row, col)
    demand = add_curve(fig, labor_demand(L_grid, 90.0 + shift_d, 0.4), "Lᵈ′", BLUE, "solid", row, col, start=True)
    add_curve(fig, labor_supply(L_grid, 30.0, 0.6), "Lˢ", GRAY, "dash", row, col, avoid=supply)
    add_curve(fig, labor_demand(L_grid, 90.0, 0.4), "Lᵈ", GRAY, "dash", row, col, start=True, avoid=demand)
    add_point(fig, L_1, w_1, "*", row, col)
    if shift_d > 0 or shift_s > 0:                               # with no shifts, ** is the same point as *
        add_point(fig, L_2, w_2, "**", row, col)
    return w_1, w_2

> **Python note.** `make_subplots(rows=1, cols=2)` creates a figure with two panels side by side, and
> `row=..., col=...` tells each drawing function which panel to use.

In [85]:
def rising_return(shift_d=15.0, shift_s=10.0):
    """Fig. 7.9: (a) the high school market is unchanged; (b) college demand shifts out more than supply."""
    L_hs, w_hs = equilibrium()
    fig = make_subplots(rows=1, cols=2, subplot_titles=("(a) High school graduates", "(b) College graduates"))
    add_curve(fig, labor_supply(L_grid), "Lˢ", TEAL, "solid", 1, 1)
    add_curve(fig, labor_demand(L_grid), "Lᵈ", BLUE, "solid", 1, 1, start=True)
    add_point(fig, L_hs, w_hs, "*", 1, 1)
    w_1, w_2 = college_market(fig, shift_d, shift_s, 1, 2)
    title = f"College wage {w_1:.1f} → {w_2:.1f}; premium over high school {w_1 / w_hs - 1:.0%} → {w_2 / w_hs - 1:.0%}"
    return market_axes(fig, title, x_title="Employment", y_title="Wage")

In [86]:
interact(rising_return,
         shift_d=FloatSlider(value=15, min=0, max=20, step=2.5, description="college demand shifts out by", style=WIDE),
         shift_s=FloatSlider(value=10, min=0, max=20, step=2.5, description="college supply shifts out by", style=WIDE));

In [87]:
# @title Website version: Fig. 9 with a precomputed slider for the shift in college labor demand
slider_figure(rising_return, "shift_d", [0, 5, 10, 15, 20], start=15, label="college demand shifts out by")

**Source.** Adapted from Fig. 7.9 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

**What to notice.**

- With the default shifts, college employment rises from 60 to 85 **and** the college wage rises from 66 to
  71. The premium over high school graduates rises from 20 to 29 percent.
- Set the demand shift to 0: the extra supply alone lowers the college wage. The rising premium needs a
  large shift in demand.
- Two explanations for that shift are **skill-biased technical change** (computers and software make
  skilled workers more productive) and **globalization** (trade with countries where skilled workers are
  scarce raises their value; trade with countries rich in unskilled labor lowers the wages of the unskilled).

**Your turn.** Keep the supply shift at 10. What is the smallest demand shift that keeps the college wage
from falling? Find it with the sliders, then explain it with the slopes 0.4 and 0.6.

<details><summary>Hint</summary>

The wage stays put when the two shifts move the equilibrium wage by the same amount in opposite directions.
For these lines that happens when $0.6 \times \text{shift}_d = 0.4 \times \text{shift}_s$ (equation 11).

</details>

In [88]:
# Your turn: try it here

## Manufacturing jobs and income inequality

Three long series show how the economy has changed for different workers: the fall of manufacturing jobs, the rise of top incomes, and unequal growth across the income distribution.

### Fig. 10: U.S. employment in manufacturing

**Goal.** See the long decline of manufacturing jobs, a part of the globalization story.

The file holds monthly employment in manufacturing, in millions of workers, from the Bureau of Labor
Statistics payroll survey.

In [89]:
manuf = pd.read_csv(DATA_URL + "fig7-10.csv", parse_dates=["date"])
fig = px.area(manuf, x="date", y="manufacturing_emp_millions", hover_name="date",
              labels={"date": "Year", "manufacturing_emp_millions": "Manufacturing jobs (millions)"},
              title="Manufacturing jobs peaked in 1979 and fell sharply after 2000")
fig.update_traces(line=dict(color=GREEN, width=2))
shade_recessions(fig, "1950-01-01")
fig.update_yaxes(range=[10, 20], tickvals=[12, 14, 16, 18, 20])      # no label in the bottom-left corner
fig.update_xaxes(rangeslider_visible=True, range=["1950-01-01", "2030-01-01"])
fig.show()

**Sources.** Data: [U.S. Bureau of Labor Statistics, Current Employment Statistics, All Employees, Manufacturing (`MANEMP`), via FRED](https://fred.stlouisfed.org/series/MANEMP);
recession dates: [NBER, U.S. Business Cycle Expansions and Contractions](https://www.nber.org/research/data/us-business-cycle-expansions-and-contractions),
via the FRED recession indicator [`USREC`](https://fred.stlouisfed.org/series/USREC).
Prepared files: [`data/ch07/fig7-10.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch07/fig7-10.csv), [`data/ch07/recessions.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch07/recessions.csv)
(see the [data notes](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch07/README.md)).
Chart: inspired by Fig. 7.10 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

**What to notice.**

- Manufacturing employment grew until the late 1970s, fell in every recession, and was roughly flat in the
  1990s.
- From 2000 to 2010 it dropped by about a third and never recovered.
- Research the textbook cites finds that rising imports from China explain about a quarter of that decline.

#### Key numbers: the fall after 2000

In [90]:
m = manuf.set_index("date").manufacturing_emp_millions
low = m.loc["2000":"2012"]
fall = m.loc["2000-01-01"] - low.min()
print(f"Peak: {m.max():.2f} million in {m.idxmax():%B %Y}")
print(f"January 2000: {m.loc['2000-01-01']:.2f} million;  low: {low.min():.2f} million in {low.idxmin():%B %Y}")
print(f"Fall: {fall:.2f} million ({fall / m.loc['2000-01-01']:.0%});  a quarter of it: {fall / 4:.2f} million jobs")
print(f"Latest: {m.iloc[-1]:.2f} million in {m.index[-1]:%B %Y}")

Peak: 19.55 million in June 1979
January 2000: 17.28 million;  low: 11.44 million in February 2010
Fall: 5.84 million (34%);  a quarter of it: 1.46 million jobs
Latest: 12.65 million in September 2026


Between January 2000 and February 2010 the sector lost 5.84 million jobs, 34 percent of its employment.
A quarter of that, the share linked to Chinese imports, is about 1.5 million jobs.

#### Beyond the United States

The ILO estimates employment in manufacturing for almost every country since 1991, from labor surveys. A
wider file holds the number of manufacturing workers (millions) and their share of all employed people.

In [91]:
manuf_world = pd.read_csv(DATA_URL + "manufacturing_countries.csv")
manuf_world[manuf_world.country == "United States"].tail(3)

,iso3,country,region,year,manufacturing_millions,manufacturing_share
6248,USA,United States,North America,2023,16.951,10.01
6249,USA,United States,North America,2024,16.404,9.66
6250,USA,United States,North America,2025,16.198,9.52


The ILO counts 16.2 million U.S. manufacturing workers in 2025, about 3.5 million more than the payroll
count above, because household surveys also count the self-employed and people whose employer is classified
elsewhere. Different sources, different levels: compare **shares** across countries, and compare levels
only within one source. A few series end before 2025 (Ukraine in 2021, Sudan in 2022).

> **Python note.** `MEASURES` is a dictionary of lists: for each choice of the `ToggleButtons`, the column
> to plot, its axis title, the usual and the wider axis top, and the unit. `column, label, high, wider, unit
> = MEASURES[measure]` unpacks the five items of one list into five variables in one line.

In [92]:
MANUF_COUNTRIES = sorted(manuf_world.country.unique())
MANUF_REGIONS = dict(zip(manuf_world.country, manuf_world.region))
MANUF_SHOWN = ["United States", "Germany", "Vietnam"]       # the website's default view
MANUF_COLORS = group_colors(MANUF_SHOWN + MANUF_COUNTRIES, region_groups(MANUF_REGIONS, MANUF_SHOWN), BOOK_COLORS)
MEASURES = {"share of jobs": ["manufacturing_share", "Manufacturing, % of all jobs", 30, 40, "points"],
            "millions of workers": ["manufacturing_millions", "Manufacturing workers (millions)", 25, 180, "million"]}


def manufacturing_countries(names=("United States",), measure="share of jobs"):
    """Fig. 7.10 for any countries: manufacturing employment (ILO), as a share of jobs or in millions."""
    chosen = manuf_world[manuf_world.country.isin(names)]
    if len(chosen) == 0:
        return go.Figure(layout=dict(title="Pick at least one country"))
    column, label, high, wider, unit = MEASURES[measure]
    by_country = chosen.groupby("country")
    change = (by_country.last()[column] - by_country.first()[column]).sort_values()
    title = f"Change since 1991 ({unit}): {change.index[0]} {round(change.iloc[0], 1) + 0.0:+.1f}"
    if len(change) > 1:
        title = title + f", {change.index[-1]} {round(change.iloc[-1], 1) + 0.0:+.1f}"
    fig = px.line(chosen, x="year", y=column, color="country", hover_name="country",
                  color_discrete_map=MANUF_COLORS, title=title,
                  labels={"year": "Year", column: label, "country": "Country"})
    if chosen[column].max() > high:
        high = wider                              # one wider range, only when a country needs it
    fig.update_yaxes(range=[0, high])
    return fig

Below, Colab shows a country picker and a measure toggle. On the website, the chart holds every country (shares
of jobs): **pick a region in the menu, then click a country in the legend to add it.**

In [93]:
interact(manufacturing_countries,
         names=SelectMultiple(options=MANUF_COUNTRIES, value=("United States",), rows=10,
                              description="countries", style=WIDE),
         measure=ToggleButtons(options=["share of jobs", "millions of workers"], value="share of jobs",
                               description="measure", style=WIDE));

In [94]:
# @title Website version: Fig. 10 for every country (pick a region, then click a country in the legend)
fig = manufacturing_countries(tuple(MANUF_COUNTRIES), "share of jobs")
region_menu(fig, MANUF_REGIONS, shown=MANUF_SHOWN, label="U.S., Germany, Vietnam")
fig.update_layout(title="Manufacturing's share of jobs fell in rich countries and rose in Vietnam")
fig.update_yaxes(autorange=True)              # the axis adapts to the countries you add
fig

In [95]:
shares = manuf_world.pivot(index="year", columns="country", values="manufacturing_share")
print(shares.loc[[1991, 2000, 2025], ["United States", "Germany", "Japan", "United Kingdom", "China", "Vietnam"]])
print(manuf_world[manuf_world.year == 2025].sort_values("manufacturing_millions").tail(3)[["country", "manufacturing_millions"]])

country  United States  Germany  Japan  United Kingdom  China  Vietnam
year                                                                  
1991             16.00    27.90  23.64           21.79  18.39     5.64
2000             13.78    23.76  20.09           16.91  19.64     8.93
2025              9.52    18.01  15.59            8.00  21.01    23.95
        country  manufacturing_millions
2694  Indonesia                  19.742
2659      India                  71.317
1189      China                 154.959


**Sources.** Data: [International Labour Organization, ILOSTAT](https://ilostat.ilo.org/data/), modelled estimates of employment
by economic activity (`EMP_2EMP_SEX_ECO_NB_A`, manufacturing and total).
Prepared file: [`data/ch07/manufacturing_countries.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch07/manufacturing_countries.csv)
(see the [data notes](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch07/README.md)).
Chart: inspired by Fig. 7.10 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

**What to notice.**

- Add **Germany**, **Japan** and the **United Kingdom**. Manufacturing's share of jobs fell in all of them
  since 1991: from 27.9 to 18.0 percent in Germany, from 23.6 to 15.6 in Japan and from 21.8 to 8.0 in the
  U.K. (the U.S.: 16.0 to 9.5). The decline is not only an American story.
- Add **China** and **Vietnam**, and switch to millions of workers. China's factories employ about 155
  million people, almost ten times as many as American ones, and Vietnam's share of jobs in manufacturing
  rose from 5.6 to 24.0 percent.

**Your turn.** How many manufacturing jobs were lost between the 1979 peak and January 2000? Compare it
with the fall after 2000.

<details><summary>Hint</summary>

`m.max() - m.loc["2000-01-01"]` gives the first fall in millions.

</details>

In [96]:
# Your turn: try it here

### Fig. 11: Income inequality in the United States and France

**Goal.** Follow a century of top incomes in two countries.

The file gives the share of national income (before taxes and benefits) received by the richest 0.1
percent of adults, the top 1 in 1,000, from the World Inequality Database. These data come from tax records,
which show the very top of the distribution well.

In [97]:
top = pd.read_csv(DATA_URL + "fig7-11.csv")
top = top.rename(columns={"top01_share_us": "United States", "top01_share_france": "France"})
top.tail()

,year,United States,France
107,2020,8.14,4.20
108,2021,9.33,5.56
109,2022,9.93,4.86
110,2023,9.93,NaN
111,2024,9.93,NaN


Look at the last rows. France stops in 2022, and the U.S. value is exactly 9.93 in 2022, 2023 and 2024. Three
identical values in a row are unusual for an economic series: the database seems to repeat its last estimate
until new tax data arrive. We keep the rows but treat 2023–2024 with care.

In [98]:
fig = px.line(top, x="year", y=["United States", "France"], hover_name="year",
              labels={"year": "Year", "value": "Income share of the top 0.1% (percent)", "variable": "Country"},
              title="Top incomes fell after 1929 in both countries, but rose again only in the U.S.")
fig.update_yaxes(range=[0, 12])
fig.update_layout(hovermode="x unified")
fig.show()

**Sources.** Data: [World Inequality Database (WID)](https://wid.world/data/), share of pre-tax national income of the top 0.1 percent,
via [Our World in Data](https://ourworldindata.org/economic-inequality); WID extends the tax-record series of
Piketty and Saez (2003).
Prepared file: [`data/ch07/fig7-11.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch07/fig7-11.csv)
(see the [data notes](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch07/README.md)).
Chart: inspired by Fig. 7.11 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

**What to notice.**

- From 1913 to 1929 the top 0.1 percent received about 9 percent of all income on average in both
  countries, and more than 10 percent in peak years (the textbook says about 10 percent).
- The Great Depression and World War II cut their share sharply, and it stayed low until about 1980.
- After 1980 the U.S. share rose back to almost 10 percent, while France's stayed much lower.

#### Key numbers: three eras

In [99]:
for start, end in [[1913, 1929], [1950, 1959], [1960, 1979], [2015, 2022]]:
    era = top[(top.year >= start) & (top.year <= end)]
    print(f"{start}-{end}: U.S. {era['United States'].mean():.1f}%, France {era['France'].mean():.1f}%")

1913-1929: U.S. 8.7%, France 9.4%
1950-1959: U.S. 5.0%, France 3.7%
1960-1979: U.S. 3.9%, France 3.3%
2015-2022: U.S. 8.8%, France 4.3%


The 1950s average is 5.0 percent in the United States and 3.7 in France; the textbook's "around 4
percent" fits the 1960s and 1970s better. Today's data also differ from the textbook's figure in two
places: a U.S. spike in 1950 (7.25 percent) and a higher French share after 2015 (5.56 percent in 2021,
against about 2.8 in the textbook). The database is revised as new tax records are added.

#### Beyond two countries

The World Inequality Database (WID) publishes the same top income shares for many countries. A wider file
holds the share of the top 0.1 percent for 58 countries: the 38 OECD members and 20 large emerging
economies.

In [100]:
top_world = pd.read_csv(DATA_URL + "top01_share_countries.csv")
top_world[top_world.country == "Brazil"].head(12)

,iso3,country,region,year,top01_share
346,BRA,Brazil,Latin America & Caribbean,1820,9.70
347,BRA,Brazil,Latin America & Caribbean,1850,9.77
348,BRA,Brazil,Latin America & Caribbean,1880,9.84
349,BRA,Brazil,Latin America & Caribbean,1900,9.90
350,BRA,Brazil,Latin America & Caribbean,1910,10.03
351,BRA,Brazil,Latin America & Caribbean,1920,10.69
352,BRA,Brazil,Latin America & Caribbean,1930,12.50
353,BRA,Brazil,Latin America & Caribbean,1940,15.00
354,BRA,Brazil,Latin America & Caribbean,1950,12.50
355,BRA,Brazil,Latin America & Caribbean,1960,9.52


**Look at your data.** Every country has a value for every year since 1980. Before 1980, about 30 countries
have long series from tax records, like the United States and France. The others, like Brazil above, have
only a few *benchmark* years from WID's historical estimates (1820, 1850, 1880, 1900, 1910, ...), and a
line chart joins them with straight segments that hide everything in between. In the latest year, WID
repeats its previous estimate for 56 of the 58 countries. The chart starts in 1900 and marks each data point.

In [101]:
TOP_COUNTRIES = sorted(top_world.country.unique())
TOP_REGIONS = dict(zip(top_world.country, top_world.region))
BOOK_PAIR = ("France", "United States")
TOP_COLORS = group_colors(TOP_COUNTRIES, region_groups(TOP_REGIONS, BOOK_PAIR), BOOK_COLORS)


def top_share_countries(names=BOOK_PAIR):
    """Fig. 7.11 for any of 58 countries: income share of the top 0.1% since 1900 (WID)."""
    chosen = top_world[top_world.country.isin(names) & (top_world.year >= 1900)]
    if len(chosen) == 0:
        return go.Figure(layout=dict(title="Pick at least one country"))
    last = chosen.year.max()
    latest = chosen[chosen.year == last].set_index("country").top01_share.sort_values()
    title = f"Top 0.1% income share in {last}: {latest.index[-1]} {latest.iloc[-1]:.1f}%"
    if len(latest) > 1:
        title = title + f" (highest), {latest.index[0]} {latest.iloc[0]:.1f}% (lowest)"
    fig = px.line(chosen, x="year", y="top01_share", color="country", hover_name="country", markers=True,
                  color_discrete_map=TOP_COLORS, title=title,
                  labels={"year": "Year", "top01_share": "Income share of the top 0.1% (percent)", "country": "Country"})
    fig.update_traces(marker=dict(size=4))
    high = 16 if chosen.top01_share.max() <= 16 else 25           # widen only for very high shares
    fig.update_yaxes(range=[0, high])
    fig.update_xaxes(range=[1898, 2026], tickvals=[1920, 1940, 1960, 1980, 2000, 2020])      # no label at the bottom-left corner
    return fig

Below, Colab shows a country picker. On the website, the chart holds all 58 countries: **pick a region in the
menu, then click a country in the legend to add it.**

In [102]:
interact(top_share_countries,
         names=SelectMultiple(options=TOP_COUNTRIES, value=BOOK_PAIR, rows=10, description="countries", style=WIDE));

In [103]:
# @title Website version: Fig. 11 for 58 countries (pick a region, then click a country in the legend)
fig = top_share_countries(tuple(TOP_COUNTRIES))
region_menu(fig, TOP_REGIONS, shown=list(BOOK_PAIR))
fig.update_layout(title="Top 0.1% shares fell in most countries until 1980, and rose in 40 of 58 since")
fig.update_yaxes(autorange=True)              # the axis adapts to the countries you add
fig

In [104]:
top_wide = top_world.pivot(index="year", columns="country", values="top01_share")
print(top_wide.loc[[1913, 1950, 1980, 2023], ["United Kingdom", "Japan", "China", "India", "Russia"]])

country  United Kingdom  Japan  China  India  Russia
year                                                
1913              14.10   7.62    NaN    NaN     NaN
1950               3.84   3.47   0.85   4.93    1.30
1980               1.78   2.26   2.39   2.05    0.60
2023               5.02   1.94   7.88   9.78    9.83


**Sources.** Data: [World Inequality Database (WID)](https://wid.world/data/), bulk download, variable `sptincj992`
(pre-tax national income share), percentile `p99.9p100` (the top 0.1 percent).
Prepared file: [`data/ch07/top01_share_countries.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch07/top01_share_countries.csv)
(see the [data notes](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch07/README.md)).
Chart: inspired by Fig. 7.11 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

**What to notice.**

- Add the **United Kingdom**: the same U-shape as the United States, from 14.1 percent in 1913 to 1.8 percent
  in 1980 and 5.0 percent in 2023.
- Add **Japan**: like France, no return to the old inequality. Its share fell from 7.6 percent in 1913 to 2.3
  in 1980, and it was 1.9 percent in 2023.
- Add **China**, **India** and **Russia**: their shares rose fastest after 1980, as markets opened (China from
  2.4 to 7.9 percent, India from 2.1 to 9.8, Russia from 0.6 to 9.8).

**Your turn.** In which year was the U.S. top 0.1 percent share lowest, and how high was it?

<details><summary>Hint</summary>

`top["United States"].idxmin()` gives the row label of the smallest value; read the year with
`top.loc[row, "year"]`.

</details>

In [105]:
# Your turn: try it here

### Fig. 12: Economic growth by income percentile

**Goal.** See that growth since 1980 has been very unequal across the income distribution.

Each point is the average annual growth of real income at one **percentile** of the distribution over 34
years: from 1946 to 1980, and from 1980 to 2014. Income is measured after taxes, transfers and noncash
benefits, from the Distributional National Accounts of Piketty, Saez and Zucman. Above the 99th percentile
the file zooms in: 99.1, 99.2, ..., 99.99, 99.991, ..., 99.999.

In [106]:
growth = pd.read_csv(DATA_URL + "fig7-12.csv")
growth.head()

,year,start_year,series,percentile,group,growth_pct
0,1980,1946,g-percentile (PSZ 2018),1.0,NaN,5.988
1,1980,1946,g-percentile (PSZ 2018),2.0,NaN,4.400
2,1980,1946,g-percentile (PSZ 2018),3.0,NaN,3.825
3,1980,1946,g-percentile (PSZ 2018),4.0,NaN,3.476
4,1980,1946,g-percentile (PSZ 2018),5.0,NaN,3.295


`series` tells the two kinds of rows apart: growth at each percentile (the textbook's two curves), and
growth of group averages (bottom 50 percent, middle 40 percent, top 10 percent, ...) from the authors'
2022 update, which also covers 1985–2019. As in the textbook, we start at the 5th percentile and stop the
1946–1980 curve at the 99th, because above it the curve jumps up and down between 1 and 3.6 percent.

> **Python note.** `|` means "or". `.map({...})` replaces each end year with a readable label, and
> `fig.add_traces(...)` adds the traces of a second chart to the first one.

In [107]:
curves = growth[(growth.series == "g-percentile (PSZ 2018)") & (growth.percentile >= 5)].copy()
curves = curves[(curves.year == 2014) | (curves.percentile <= 99)]
curves["period"] = curves.year.map({1980: "1946–1980 (“In 1980”)", 2014: "1980–2014 (“In 2014”)"})
groups = growth[growth.series == "group average (PSZ 2022)"].copy()
groups["period"] = groups.start_year.astype(str) + "–" + groups.year.astype(str) + ", group averages"
fig = px.line(curves, x="percentile", y="growth_pct", color="period", markers=True, hover_name="period",
              color_discrete_sequence=["#e0a33b", BLUE],
              labels={"percentile": "Income percentile", "growth_pct": "Average annual growth (percent)", "period": "Period"},
              title="Since 1980, the richest have seen the fastest income growth")
dots = px.scatter(groups, x="percentile", y="growth_pct", color="period", hover_name="group",
                  color_discrete_sequence=["#5aa0d8", TEAL, GREEN], symbol_sequence=["diamond"])
fig.add_traces(dots.data)
fig.update_traces(marker=dict(size=6))
fig.update_yaxes(range=[-0.5, 7])
fig.update_layout(legend=dict(orientation="h", y=-0.2, yanchor="top"))
fig.show()

**Sources.** Data: Piketty, Saez and Zucman (2018), "Distributional National Accounts: Methods and Estimates for the United
States," *Quarterly Journal of Economics* 133(2), [doi:10.1093/qje/qjx043](https://doi.org/10.1093/qje/qjx043): main data file, sheet `TS11(NYT)`, and
the February 2022 appendix tables II, sheet `TC3`, both from the authors' [Distributional National Accounts page](https://gabriel-zucman.eu/usdina/).
Prepared file: [`data/ch07/fig7-12.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch07/fig7-12.csv)
(see the [data notes](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch07/README.md)).
Chart: inspired by Fig. 7.12 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

**What to notice.**

- From 1946 to 1980 (orange), the poorer half grew fastest: above 2.5 percent a year at the bottom, under 2
  percent in the upper middle.
- From 1980 to 2014 (blue), the pattern flipped: under 1 percent a year up to the 57th percentile,
  above 2 percent at the 99th percentile, and about 6 percent at the very top.
- The group averages (diamonds) tell the same story. The newest period, 1985–2019, still favors the top,
  though less steeply.

#### Key numbers: who grew how fast

In [108]:
p1980 = curves[curves.year == 1980].set_index("percentile").growth_pct
p2014 = growth[(growth.series == "g-percentile (PSZ 2018)") & (growth.year == 2014)].set_index("percentile").growth_pct
print(f"Median (50th percentile): {p1980[50]:.2f}% a year in 1946-1980, {p2014[50]:.2f}% in 1980-2014")
print("Percentiles with falling income, 1980-2014:", p2014[p2014 < 0].index.tolist())
faster = p2014[(p2014.index >= 5) & (p2014.index <= 99)] > p1980
print(f"First percentile where 1980-2014 growth beats 1946-1980: {faster[faster].index.min():.0f}")
print(f"Top of the distribution, 1980-2014: 99.9th {p2014[99.9]:.2f}%, 99.99th {p2014[99.99]:.2f}%, 99.999th {p2014[99.999]:.2f}%")

Median (50th percentile): 1.93% a year in 1946-1980, 0.91% in 1980-2014
Percentiles with falling income, 1980-2014: [1.0, 2.0, 3.0, 4.0, 5.0]
First percentile where 1980-2014 growth beats 1946-1980: 96
Top of the distribution, 1980-2014: 99.9th 3.24%, 99.99th 4.04%, 99.999th 5.96%


The textbook says that average growth was about 2 percent a year before 1980 and 1.4 percent after. Our file
does not have the overall average, but the median tells the story: 1.93 percent a year before 1980 and 0.91
after. Only the bottom five percentiles lost income after 1980, and the 1980–2014 curve rises above the old
one from the 96th percentile on. At the very top, growth reached 4 to 6 percent a year, as fast as the
fastest-growing countries.

#### Beyond the United States

WID estimates the whole income distribution of every country, so we can draw the same "growth by
percentile" curve elsewhere. A wider file holds it for the 58 countries of Fig. 11, for the textbook's
34-year window, 1980–2014, and for the latest 34 years, 1990–2024.

In [109]:
pct_world = pd.read_csv(DATA_URL + "percentile_growth_countries.csv")
pct_world.head()

,iso3,country,region,period,percentile,growth_pct
0,ARG,Argentina,Latin America & Caribbean,1980–2014,1.0,3.64
1,ARG,Argentina,Latin America & Caribbean,1980–2014,2.0,3.64
2,ARG,Argentina,Latin America & Caribbean,1980–2014,3.0,3.64
3,ARG,Argentina,Latin America & Caribbean,1980–2014,4.0,3.64
4,ARG,Argentina,Latin America & Caribbean,1980–2014,5.0,3.64


Two differences from the textbook's curve. First, WID measures income **before** taxes and transfers for
every country, while the textbook's curve is after them. For the United States in 1980–2014, pre-tax income
at the 10th percentile *fell* by 1.25 percent a year, while post-tax income rose by 0.26 percent: taxes and
transfers lift the bottom. At the top the two measures agree. Second, WID's numbers for countries with poor
tax data are partly model estimates, and its last year (2024) extends 2023 forward. Treat small differences
between countries with care.

**Look at your data.** In most countries WID gives the poorest 5 percent of adults no pre-tax income at all
and fills the next few percentiles with a smooth model, so their "growth rates" are growth rates of tiny
numbers. The chart therefore starts at the 10th percentile (the textbook's starts at the 5th).

> **Python note.** `.isin(names) & (pct_world.period == period)` combines two conditions: both must be true.
> The `for` loop tries the three axis ranges in turn and stops (`break`) at the first one that holds every
> point.

In [110]:
PCT_COUNTRIES = sorted(pct_world.country.unique())
PCT_REGIONS = dict(zip(pct_world.country, pct_world.region))
PCT_SHOWN = ["United States", "China"]                      # the website's default view
PCT_COLORS = group_colors(PCT_SHOWN + PCT_COUNTRIES, region_groups(PCT_REGIONS, PCT_SHOWN), BOOK_COLORS)


def growth_by_percentile(names=("United States",), period="1980–2014"):
    """Fig. 7.12 for any of 58 countries: growth of pre-tax income per adult at each percentile (WID)."""
    chosen = pct_world[pct_world.country.isin(names) & (pct_world.period == period) & (pct_world.percentile >= 10)]
    if len(chosen) == 0:
        return go.Figure(layout=dict(title="Pick at least one country"))
    median = chosen[chosen.percentile == 50].set_index("country").growth_pct.sort_values()
    title = f"Growth of median income, {period}: {median.index[-1]} {round(median.iloc[-1], 1) + 0.0:.1f}% a year"
    if len(median) > 1:
        title = title + f", {median.index[0]} {round(median.iloc[0], 1) + 0.0:.1f}%"
    fig = px.line(chosen, x="percentile", y="growth_pct", color="country", hover_name="country", markers=True,
                  color_discrete_map=PCT_COLORS, title=title,
                  labels={"percentile": "Income percentile", "growth_pct": "Average annual growth (percent)",
                          "country": "Country"})
    fig.update_traces(marker=dict(size=4))
    for low, high in [(-2, 7), (-6, 12), (-14, 18)]:            # widen only when needed
        if chosen.growth_pct.min() >= low and chosen.growth_pct.max() <= high:
            break
    fig.update_layout(yaxis_range=[low, high], xaxis_range=[0, 101])
    fig.update_xaxes(tickvals=[10, 20, 30, 40, 50, 60, 70, 80, 90, 100])      # no label at the bottom-left corner
    return fig

Below, Colab shows a country picker and both periods, 1980–2014 and 1990–2024. On the website, the chart holds
all 58 countries for the textbook's window, 1980–2014: **pick a region in the menu, then click a country in the
legend to add it.**

In [111]:
interact(growth_by_percentile,
         names=SelectMultiple(options=PCT_COUNTRIES, value=("United States",), rows=10,
                              description="countries", style=WIDE),
         period=ToggleButtons(options=["1980–2014", "1990–2024"], value="1980–2014", description="period", style=WIDE));

In [112]:
# @title Website version: Fig. 12 for 58 countries, 1980-2014 (pick a region, then click a country)
fig = growth_by_percentile(tuple(PCT_COUNTRIES), "1980–2014")
region_menu(fig, PCT_REGIONS, shown=PCT_SHOWN, label="U.S. and China")
fig.update_layout(title="1980–2014: in China every percentile grew at least 3% a year")
fig.update_yaxes(autorange=True)              # the axis adapts to the countries you add
fig

In [113]:
period_1980 = pct_world[pct_world.period == "1980–2014"].pivot(index="country", columns="percentile", values="growth_pct")
period_1990 = pct_world[pct_world.period == "1990–2024"].pivot(index="country", columns="percentile", values="growth_pct")
print(period_1980.loc[["United States", "France", "China", "India"], [10, 50, 90, 99, 99.99]])
print(period_1990.loc[["United States", "Brazil"], [10, 13, 50, 99.99]])
print("Colombia, 18th percentile, 1980-2014:", period_1980.loc["Colombia", 18])

percentile     10.00  50.00  90.00  99.00  99.99
country                                         
United States  -1.25   0.58   1.59   2.10   4.08
France          0.72   0.88   1.04   1.46   2.19
China           3.29   3.87   6.00   6.30   7.83
India           1.69   1.71   1.91   5.06   7.34
percentile     10.00  13.00  50.00  99.99
country                                  
United States   0.53   0.73   0.65   2.71
Brazil         -3.74 -12.55   0.75  -0.27
Colombia, 18th percentile, 1980-2014: 17.19


**Sources.** Data: [World Inequality Database (WID)](https://wid.world/data/), bulk download, variable `aptincj992`
(average pre-tax national income per adult) at each g-percentile.
Prepared file: [`data/ch07/percentile_growth_countries.csv`](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch07/percentile_growth_countries.csv)
(see the [data notes](https://github.com/quarcs-lab/macro102-site/blob/main/data/ch07/README.md)).
Chart: inspired by Fig. 7.12 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

**What to notice.**

- Add **France**: in 1980–2014 its curve is almost flat, between 0.7 and 2.2 percent a year from the 5th
  percentile to the very top. The steep U.S. curve is not the only possible pattern.
- Add **China** and **India**: every percentile from the 10th up grew faster than in the United States, 3 to
  9 percent a year in China, but the top still grew fastest. Fast growth can lift all incomes and raise inequality at the
  same time.
- Switch to **1990–2024**: the U.S. curve is flatter and positive everywhere (0.5 percent at the 10th
  percentile, 2.7 at the 99.99th).
- Add **Brazil** and **Colombia**: near the bottom their curves jump to −12.5 percent (Brazil, 13th
  percentile, 1990–2024) or +17 percent a year (Colombia, 18th percentile, 1980–2014). Such jumps come from
  very small incomes in the starting year, not from real booms or collapses.

**Your turn.** Use the `groups` table: how fast did the average income of the bottom 50 percent grow in
1980–2014, and how fast in 1985–2019?

<details><summary>Hint</summary>

Keep the rows with `groups.group == "bottom 50%"` and look at the `year` and `growth_pct` columns.

</details>

In [114]:
# Your turn: try it here

## Back to the 1970s: a falling college premium

We close with the same college labor market used in reverse, to explain a decade in which the return to college went down.

### Fig. 13: The declining college wage premium of the 1970s

**Goal.** Use the college labor market of Fig. 9 to explain why the premium *fell* in the 1970s.

The textbook's data show the college premium falling during the 1970s, even though technology kept raising
the demand for college graduates. The answer: in that decade supply shifted out by **more** than demand. The
baby boom generation reached college age, and the Vietnam War drew many young men to college. We reuse
`college_market`, now with a large supply shift.

In [115]:
def premium_1970s(shift_d=10.0, shift_s=25.0):
    """Fig. 7.13: college labor supply shifts out by more than demand, so the college wage falls."""
    fig = go.Figure()
    w_1, w_2 = college_market(fig, shift_d, shift_s)
    L_1 = equilibrium(a_d=90.0, b_d=0.4, a_s=30.0, b_s=0.6)[0]
    L_2 = equilibrium(a_d=90.0 + shift_d, b_d=0.4, a_s=30.0 - shift_s, b_s=0.6)[0]
    return market_axes(fig, f"College employment {L_1:.1f} → {L_2:.1f}, college wage {w_1:.1f} → {w_2:.1f}",
                       x_title="Employment of college graduates", y_title="Wage of college graduates")

In [116]:
interact(premium_1970s,
         shift_d=FloatSlider(value=10, min=0, max=20, step=2.5, description="college demand shifts out by", style=WIDE),
         shift_s=FloatSlider(value=25, min=0, max=30, step=2.5, description="college supply shifts out by", style=WIDE));

In [117]:
# @title Website version: Fig. 13 with a precomputed slider for the shift in college labor supply
slider_figure(premium_1970s, "shift_s", [0, 5, 10, 15, 20, 25, 30], start=25, label="college supply shifts out by")

**Source.** Adapted from Fig. 7.13 in Jones (2024), [*Macroeconomics*, 6th ed.](https://wwnorton.com/books/9781324063612)

**What to notice.**

- Demand shifts out by 10 and supply by 25: employment of college graduates rises from 60 to 95, but their
  wage falls from 66 to 62.
- Figs. 9 and 13 use the same model. Whether the premium rises or falls depends only on which shift wins.
- Move the supply shift down to 15 and the wage stops falling: $0.6 \times 10 = 0.4 \times 15$.

**Your turn.** With a supply shift of 25, how large must the demand shift be for the college wage to stay at 66?

<details><summary>Hint</summary>

Use the rule from Fig. 9 (equation 11): the wage stays put when $0.6 \times \text{shift}_d = 0.4 \times \text{shift}_s$.
Check with `premium_1970s(shift_d=..., shift_s=25)`.

</details>

In [118]:
# Your turn: try it here

## Concluding remarks

- The employment-population ratio rose for decades as women joined the labor force, while men's ratio
  fell. Both the ratio and the unemployment rate swing sharply in recessions, and identity (1),
  $E/N = (1-u)\,LF/N$, shows why the two statistics can tell different stories. Equation (2) splits
  unemployment into a natural rate and a cyclical part.
- In the supply-and-demand model of the labor market, equation (3) gives the equilibrium wage and
  employment. An income tax shifts supply back and lowers employment (equation 4); a fall in labor demand
  lowers both the wage and employment. If the wage does not fall, employment falls even more and some
  people who want a job cannot find one (equation 5).
- The bathtub model, equation (6), gives the natural rate of unemployment, $u^* = \bar{s}/(\bar{f} + \bar{s})$
  (equation 7): about 5 percent with U.S. job finding and separation rates. Unemployment returns to it
  quickly after a shock (equation 8). Europe's higher unemployment since the 1980s reflects adverse shocks
  combined with labor market institutions.
- Present discounted values, equations (9) and (10), compare payments at different dates: a working life is
  worth millions of dollars today, and a college degree several hundred thousand (equation 12).
- The college premium rose while college graduates became more numerous, so demand for skills must have
  grown faster than supply (equation 11). Inequality at the very top has risen sharply in the United States
  since 1980.
- The next chapter turns to the other long-run topic of the book: **inflation**.